# Koebuta Colab v1.00 — Irodori-TTS v4.1 朗読スタジオ

[GitHub：ソース・使い方・更新情報](https://github.com/karin-tsumugi-lab/koebuta-colab)

Koebuta独自部分は[用途制限付きライセンス](https://github.com/karin-tsumugi-lab/koebuta-colab/blob/main/LICENSE)で提供します。商用利用・改変・再配布は可能ですが、本人の同意がない声の複製・なりすまし、詐欺、脅迫・嫌がらせ等への利用は禁止します。[第三者の利用条件](https://github.com/karin-tsumugi-lab/koebuta-colab/blob/main/THIRD_PARTY_NOTICES.md)も確認してください。

チャットLLMと対話して作った**シナリオJSON**を読み込み、声を設定し、台本の編集・試聴から完成WAVまで扱うノートブックです。画面は左に準備、中央に台本、右に生成・保存の３領域です。

## ⚠️ 最重要：Google Colab の無料枠で外部Web UIを使わないでください

**このノートブックは、Gradioの外部Web UIを使ってColabのGPU上で音声を生成します。Google Colabの無料の管理対象ランタイムでは、ノートブックUIを迂回して主にWeb UIを操作する行為が制限されており、予告なくランタイムが終了する可能性があります。**

- Colab上で利用する場合は、**有料プランを契約し、コンピューティング単位の残量がプラスの状態**を推奨します。プランの購入だけで無制限・無条件の利用が保証されるわけではありません。
- これは単なる処理時間の不足とは別の**利用ポリシー上の制限**です。知らずに利用しても、ランタイム終了や利用制限につながる可能性があります。Googleアカウント自体の永久停止を意味するとは限りません。
- 規約は変更されることがあるため、利用前に[Google Colab公式FAQ](https://research.google.com/colaboratory/faq.html?hl=ja)と[Colab追加利用規約](https://research.google.com/colaboratory/intl/ja/tos_v6.html)をお読みください。
- Colabのランタイムは一時環境です。**切断やセッション終了でJSON・WAV・参照音声・キャッシュが失われる場合があります。** 必要なファイルは必ず手元に保存してください。
- 外部共有URLはインターネットからアクセス可能です。認証を有効にしていますが、**URLとパスワードを公開しないでください。** 他人の原稿や音声を不用意にアップロードしないでください。

## 重要：Irodori-TTSと生成音声の倫理的な利用

本ノートブックは [Irodori-TTS（Aratako氏）](https://github.com/Aratako/Irodori-TTS) と、公開モデル [v4.1-Small](https://huggingface.co/Aratako/Irodori-TTS-v4.1-Small) を利用します。モデルはMITライセンスで提供されていますが、**モデルカードにはライセンスとは別に倫理的な利用制限**があります。

- **本人の明示的な同意のない声のクローンや、本人であるかのようになりすます行為は行わないでください。** 著名人・声優だけでなく一般人も含みます。
- 人を欺く目的のディープフェイクや誤情報の発信に利用しないでください。参照音声を使わなくても、偶然実在の声に似る可能性があります。
- 原稿、参照音声、生成物を利用・公開できる権限があるか確認し、関係する法令・利用条件を守ってください。利用者自身が責任を持って判断してください。

## 重要：電子透かしと結合音声

- Irodori-TTSは **SilentCipherの依存ライブラリとモデルファイルが利用できる場合に**、生成した音声へ電子透かしを埋め込みます。**すべての環境・出力に必ず透かしが付くという保証ではありません。** 意図的な透かしの除去・改ざんは避けてください。
- 長尺の結合WAVで透かしを確実に検出・照合できるかは、このノートブックでは検証していません。出所の確認が必要な場合は、結合前の個別WAVと、可能なら生成条件・原稿を保管してください。**透かしを権利や出自の唯一の証明としないでください。**

## Irodori-TTS v4.1 の音声生成上の注意

- **極端に短い文章は、読み飛ばし・不自然な発音などが生じる場合があります。** タイトルや著者名、一言だけの台詞は、実際に試聴して確認してください。必要なら同じ話者の隣接発話との結合を検討してください。
- 難読漢字、固有名詞、特殊なルビは読み間違えることがあります。原文 `text` は保持し、読み上げ文 `tts_text` だけを調整できます。
- **日本語専用モデル**です。参照音声は、できれば合計30秒程度以上の明瞭な音声を推奨します。
- 参照音声と声・演技の指示が矛盾すると生成品質が不安定になる場合があります。複雑な指示や感情絵文字が、毎回同じ効果になる保証はありません。

## 使い方

1. ColabでGPUランタイムを選び、ノートブックのセルを上から順に実行します。
2. 起動セルの出力に表示される**ログインユーザー名とパスワード**を使い、Gradioの共有リンクにログインします。
3. 左でJSONの読み込みと参照音声の割り当て、中央で発話の確認・修正、右で試聴・全編音声の生成・保存を行います。
4. **終了前に、完成WAV・編集後JSON・必要な個別音声と参照音声を保存してください。**

固定の話者数・文字数上限はJSON仕様には設けていませんが、Colabの実際のメモリ・GPU・操作時間には制約があります。未解決の話者IDや原文欠落があれば音声生成前に確認を求めます。


## v1.00の発話長設定

初期設定では発話の区切りを変更しません。長さの赤い警告は目安であり、実際の声・話速・演技の影響を受けます。中央の『発話の長さを確認・調整』で、候補を確認したうえで明示的に適用できます。感情と演技の反映設定は右カラムの上部に常時表示しています。

個別発話の分割ボタン、一括調整の確認チェック、個別結合時の感情・演技の優先規則を追加しました。


## 1. セットアップ（GPU確認・Irodori導入・モデルの取得）

既定ではIrodori-TTSの公開リポジトリから `main` を取得し、実際のコミットIDを表示します。再現性が必要な場合は `KOEBUTA_IRODORI_COMMIT` を指定してください。
依存関係の導入後にランタイム再起動が必要になった場合は、再起動後にセルを順番に再実行してください。**再実行前に未保存の成果物がないか確認してください。**


In [ ]:
import subprocess, sys, os, shutil, re
from pathlib import Path

# 指定したい場合のみ環境変数でコミットを固定。未指定なら最新版mainを使用し、実際のコミットを記録します。
# 既存の旧コミットはv4.1との互換性を確認できなかったため、ここでは固定しません。
IRODORI_COMMIT = os.environ.get('KOEBUTA_IRODORI_COMMIT','').strip()
IRODORI_DIR = Path('/content/Irodori-TTS')
MODEL_ID = 'Aratako/Irodori-TTS-v4.1-Small'
MODEL_REVISION = '2b28324dc263ed5e6638b3cf3dd94c82ead07b4b'
CODEC_ID = 'Aratako/Semantic-DACVAE-Japanese-32dim'
CODEC_REVISION = '47376ee24834d7a05a48ebabfe3cde29b3c5e214'

# GPU確認
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True)
print(r.stdout.strip() or 'GPUが見えません。ランタイム > ランタイムのタイプを変更 で GPU を選んでください。')
assert r.returncode == 0, 'GPUランタイムで実行してください'

# 既存のフォルダーは削除せず、別名に退避する必要がある場合は作業を中止
if IRODORI_DIR.exists() and not (IRODORI_DIR / '.git').is_dir():
    raise RuntimeError(f'{IRODORI_DIR} はGit管理外です。手動で確認してください（自動削除しません）')
if not IRODORI_DIR.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/Aratako/Irodori-TTS.git', str(IRODORI_DIR)])
# 利用者の変更がある既存リポジトリを上書きしない
status = subprocess.check_output(['git','-C',str(IRODORI_DIR),'status','--porcelain'],text=True).strip()
if status:
    raise RuntimeError('既存のIrodori-TTSに未コミット変更があります。変更を保護するため停止します')
target = IRODORI_COMMIT or 'main'
subprocess.check_call(['git','-C',str(IRODORI_DIR),'fetch','--depth','1','origin',target])
subprocess.check_call(['git','-C',str(IRODORI_DIR),'checkout','--detach','FETCH_HEAD'])

IRODORI_RESOLVED_COMMIT = subprocess.check_output(['git','-C',str(IRODORI_DIR),'rev-parse','HEAD'],text=True).strip()
print('Irodori-TTS:', IRODORI_RESOLVED_COMMIT)

# 取得した上流ファイルは変更せず、Python 3.13ではインストール用のコピーにだけ
# sentencepieceの互換範囲を設定する。再実行してもGit作業ツリーは汚れない。
req_file = IRODORI_DIR / 'requirements.txt'
req_copy = Path('/content/koebuta_requirements_runtime.txt')
req_text = req_file.read_text(encoding='utf-8')
if sys.version_info >= (3,13):
    req_text, changed = re.subn(r'(?m)^sentencepiece[^\n]*$', 'sentencepiece>=0.2.0', req_text)
    if changed != 1:
        print('注意: sentencepiece依存の記述形式が変わったため、Python 3.13互換調整を適用していません')
req_copy.write_text(req_text,encoding='utf-8')

# 依存を導入
print("パッケージのインストールを開始します...")

print("1. requirements.txt から依存ライブラリをインストール中...")
res_req = subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(req_copy)], capture_output=True, text=True)
if res_req.returncode != 0:
    print("=== requirements.txt インストールエラー ===")
    print(res_req.stdout)
    print(res_req.stderr)
    raise RuntimeError("requirements.txt のインストールに失敗しました")

print("2. Irodori-TTS 本体をインストール中...")
res_main = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', str(IRODORI_DIR)], capture_output=True, text=True)
if res_main.returncode != 0:
    print("=== 本体インストールエラー ===")
    print(res_main.stdout)
    print(res_main.stderr)
    raise RuntimeError("本体のインストールに失敗しました")

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gradio==6.5.1', 'regex>=2024.11.6'])

# モデルとコーデックを取得
from huggingface_hub import hf_hub_download
ckpt = hf_hub_download(MODEL_ID, 'model.safetensors', revision=MODEL_REVISION)
codec = hf_hub_download(CODEC_ID, 'weights.pth', revision=CODEC_REVISION)
print('checkpoint:', ckpt)
print('codec:', codec)
print('セットアップ完了。')


## 2. 本体ロジック（検証・読み変換・分割）

画面から使う関数群です。実行するだけで構いません。
元版 `irodori_cli` の規則（形式・感情表・ルビ変換・分割）をもとに、ノートブック用に単純化してあります。

In [ ]:
import copy, hashlib, json, math, random, re, shutil, unicodedata
from pathlib import Path

WORK = Path('/content/koebuta_work')
SEG_DIR = WORK / 'segments'      # 生成済み発話WAVとキー
VOICE_DIR = WORK / 'voices'      # 採用した参照WAV
OUT_DIR = WORK / 'out'
for d in (SEG_DIR, VOICE_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# 感情ID -> 先頭に付ける絵文字（元版 core.EMOTIONS と同じ対応）
EMOTIONS = dict(zip(
    ['neutral','joy','anger','sad','kind','tease','anxiety','tension','surprise','tired','whisper','giggle','fear','sleepy','resolve'],
    ['','😆','😠','😭','🫶','😏','🥺','😰','😲','😮‍💨','👂','🤭','😱','😪','💥']))
RUBY_FORMATS = ('none', 'aozora', 'narou', 'kakuyomu', 'pixiv')

# Web UIではGradioのセッションごとに独立した状態を使う。
# GPUのモデルだけは共有して構わないので、ランタイムは別途グローバルで管理する。
from contextvars import ContextVar
from collections.abc import MutableMapping

def new_session():
    return {'project':None, 'voices':{}, 'sample':None, 'last_files':[],
            'undo':[], 'plan_warnings':[], 'length_min':12, 'length_max':140}

_CTX = ContextVar('koebuta_session',default=None)
_FALLBACK_SESSION = new_session()  # ノートブック内からの直接呼び出し用

class SessionData(MutableMapping):
    def _data(self): return _CTX.get() if _CTX.get() is not None else _FALLBACK_SESSION
    def __getitem__(self,k): return self._data()[k]
    def __setitem__(self,k,v): self._data()[k]=v
    def __delitem__(self,k): del self._data()[k]
    def __iter__(self): return iter(self._data())
    def __len__(self): return len(self._data())

STATE = SessionData()
_RUNTIME = {'instance':None, 'model':None}
# voices: プロファイルキー -> 参照WAVパス。単独方式は 'shared'、複数方式は role の id をキーにする。

# ---------- 標準シナリオJSONの寛容な読み込み ----------
SCENARIO_KEYS = {'schema_version', 'source', 'speakers', 'segments'}
SPEAKER_KEYS = {'name', 'kind', 'bio', 'voice_caption', 'acting_description', 'age', 'gender', 'pitch', 'timbre', 'habitual_style'}
SEGMENT_KEYS = {'id', 'speaker_id', 'text', 'emotion', 'acting_description', 'tts_text',
                'pause_after_ms', 'segment_type', 'text_format', 'blank_lines_after'}
ATTR_VALUES = {
    'age': {'young_adult','adult','elderly'}, 'gender': {'feminine','masculine'},
    'pitch': {'high','mid','low'},
    'timbre': {'clear','warm','soft','husky','bright','resonant'},
    'habitual_style': {'gentle','measured','brisk','formal','casual','calm'},
}

def _valid_id(v):
    return isinstance(v, str) and bool(v.strip())

def _optional_string(v, label, warnings):
    if v is None: return ''
    if isinstance(v, str): return v
    warnings.append(f'{label} は文字列ではないため、指定なしとして扱いました')
    return ''

def _nonnegative_int(v):
    return type(v) is int and v >= 0

def _temp_id(used, i):
    candidate = f'_pending_{i}'
    while candidate in used:
        candidate += '_'
    used.add(candidate)
    return candidate

def _safe_scalar(v, default, predicate, label, warnings):
    if predicate(v): return v
    warnings.append(f'{label} が不正なため標準設定を使用しました')
    return default

def normalize_scenario(raw):
    '''受け付けた値は失わず編集用に保持。音声生成の可否はvalidate_projectで判断する。'''
    if not isinstance(raw, dict):
        raise ValueError('JSONの最上位はオブジェクトにしてください')
    legacy = raw.get('version') == 1 and 'roles' in raw and 'speakers' not in raw
    if not legacy:
        version = raw.get('schema_version')
        if version not in (None, 1) or type(version) not in (int, type(None)):
            raise ValueError(f'未対応のシナリオ形式バージョン: {version!r}')
        if version is None and not isinstance(raw.get('speakers'), dict):
            raise ValueError('形式バージョンがなく、v1のspeakersを確認できません')

    warnings = []
    if not legacy and 'schema_version' not in raw:
        warnings.append('形式バージョンがないためv1形式として読み込みました')
    roles = []
    if legacy:
        source_roles = raw.get('roles')
        if not isinstance(source_roles, list):
            warnings.append('話者一覧が不正なため、空の一覧として読み込みました')
            source_roles = []
        role_items = [(r.get('id') if isinstance(r, dict) else None, r) for r in source_roles]
    else:
        source_roles = raw.get('speakers')
        if not isinstance(source_roles, dict):
            warnings.append('話者一覧が見つからないか形式が不正です')
            source_roles = {}
        role_items = list(source_roles.items())
    used_roles = set()
    for i, (key, p) in enumerate(role_items, 1):
        if not isinstance(p, dict):
            warnings.append(f'話者#{i} の設定がオブジェクトではありません')
            p = {}
        if not _valid_id(key) or key in used_roles:
            warnings.append(f'話者#{i} のIDが欠落または重複しています。確認が必要です')
            ident = _temp_id(used_roles, i)
            pending = True
        else:
            ident = key
            used_roles.add(ident)
            pending = False
        role = {'id': ident, 'name': p.get('name') if isinstance(p.get('name'), str) and p['name'].strip() else ident,
                'kind': p.get('kind') if p.get('kind') in ('narrator','character') else None,
                'bio': '', 'caption': '', 'acting': '', '_extras': {}, '_pending_id': pending}
        if role['name'] == ident and not (isinstance(p.get('name'), str) and p['name'].strip()):
            warnings.append(f'話者 {ident}: 表示名がないため話者IDを使います')
        if role['kind'] is None:
            warnings.append(f'話者 {ident}: 種別が未確定です（地の文か登場人物かを確認してください）')
        names = [('bio','bio'), ('voice_caption','caption'), ('acting_description','acting')]
        for source_key, out_key in names:
            raw_value = p.get(source_key) if not legacy else p.get({'caption':'caption','acting':'acting'}.get(out_key, source_key))
            role[out_key] = _optional_string(raw_value, f'話者 {ident} / {source_key}', warnings)
        for attr, values in ATTR_VALUES.items():
            val = p.get(attr)
            if val in values if isinstance(val, str) else False:
                role[attr] = val
            elif val is not None:
                warnings.append(f'話者 {ident}: {attr} が不正なため無視しました')
        role['_extras'] = {k: copy.deepcopy(v) for k,v in p.items() if k not in (SPEAKER_KEYS if not legacy else SPEAKER_KEYS | {'id','caption','acting'})}
        roles.append(role)

    rows = raw.get('segments')
    if not isinstance(rows, list):
        warnings.append('台本項目が見つからないか配列ではありません')
        rows = []
    used_segments = set()
    segments = []
    for i, s in enumerate(rows, 1):
        if not isinstance(s, dict):
            warnings.append(f'台本項目#{i} がオブジェクトではありません')
            s = {}
        original_id = s.get('id')
        if not _valid_id(original_id) or original_id in used_segments:
            sid = _temp_id(used_segments, i)
            pending = True
            warnings.append(f'台本項目#{i}: IDの欠落・重複を検出。仮ID {sid} を割り当てました')
        else:
            sid = original_id
            used_segments.add(sid)
            pending = False
        segment_type = s.get('segment_type', 'speech')
        if segment_type not in ('speech', 'separator') if isinstance(segment_type, str) else True:
            warnings.append(f'{sid}: 不明な項目種別です。確認するまで読み上げません')
        emotion = s.get('emotion', 'neutral')
        if not isinstance(emotion,str) or emotion not in EMOTIONS:
            warnings.append(f'{sid}: 未知の感情を「平静」に置き換えました')
            emotion = 'neutral'
        raw_role = s.get('role') if legacy else s.get('speaker_id')
        seg = {'id': sid, 'role': raw_role if isinstance(raw_role, str) else None,
               'text': s.get('text'), 'emotion': emotion, 'acting':
               _optional_string(s.get('acting' if legacy else 'acting_description'), f'{sid} / 演技指示', warnings),
               'segment_type': segment_type, '_pending_id': pending, '_extras': {}}
        if not isinstance(seg['text'], str) and segment_type == 'speech':
            warnings.append(f'{sid}: 原文が欠落または文字列ではありません。補完せず保持しています')
        for k in ('tts_text', 'pause_after_ms', 'text_format', 'blank_lines_after'):
            if k not in s: continue
            val = s[k]
            if k == 'tts_text':
                if isinstance(val, str) and val.strip(): seg[k] = val
                else: warnings.append(f'{sid}: 無効な読み上げ文は無視しました')
            elif k in ('pause_after_ms','blank_lines_after'):
                if _nonnegative_int(val): seg[k] = val
                else: warnings.append(f'{sid}: {k} の異常値は無視しました')
            elif k == 'text_format':
                if isinstance(val,str) and val in RUBY_FORMATS: seg[k] = val
                else: warnings.append(f'{sid}: 不明なルビ形式は無視しました')
        seg['_extras'] = {k: copy.deepcopy(v) for k,v in s.items() if k not in (SEGMENT_KEYS if not legacy else SEGMENT_KEYS | {'role','acting'})}
        segments.append(seg)

    source = raw.get('source') if isinstance(raw.get('source'), str) else None
    if 'source' in raw and source is None: warnings.append('原稿全文が文字列ではないため、原文照合を省略しました')
    reading_mode = raw.get('reading', 'cast') if legacy else 'cast'
    if reading_mode not in ('shared','cast'): reading_mode = 'cast'
    project = {
        'title': raw.get('title', '新しい朗読') if legacy else '新しい朗読',
        'source': source, 'roles': roles, 'segments': segments, 'reading': reading_mode,
        'baseCaption': raw.get('baseCaption','自然で聞き取りやすい日本語の朗読。') if legacy else '自然で聞き取りやすい日本語の朗読。',
        'seed': raw.get('seed',314) if legacy else 314,
        'steps': raw.get('steps',40) if legacy else 40,
        'pause': raw.get('pause',0.28) if legacy else 0.28,
        'acting': raw.get('acting',False) if legacy else False,
        'emotion': raw.get('emotion',True) if legacy else True,
        'narratorEmotion': raw.get('narratorEmotion',False) if legacy else False,
        'textOptions': raw.get('textOptions',{}) if legacy and isinstance(raw.get('textOptions',{}),dict) else {},
        '_load_warnings': warnings,
        '_extras': {k:copy.deepcopy(v) for k,v in raw.items() if k not in (SCENARIO_KEYS if not legacy else {'version','source','roles','segments','title','reading','baseCaption','seed','steps','pause','acting','emotion','narratorEmotion','textOptions'})},
        '_legacy': legacy,
    }
    project['steps'] = _safe_scalar(project['steps'],40,lambda v:type(v) is int and v in (20,40,60),'生成ステップ数',warnings)
    project['seed'] = _safe_scalar(project['seed'],314,lambda v:type(v) is int and 0 <= v <= 2147483647,'生成シード',warnings)
    project['pause'] = _safe_scalar(project['pause'],0.28,lambda v:type(v) in (int,float) and math.isfinite(v) and 0 <= v <= 30,'標準の間',warnings)
    return project

def validate_project(project):
    '''現在の状態を動的検証する。問題はユーザーに知らせ、データは破棄しない。'''
    issues = []
    def add(level, item, message): issues.append((level, item, message))
    if not project: return [('エラー','全体','シナリオがありません')]
    role_ids = set()
    for r in project.get('roles',[]):
        rid = r.get('id')
        if not _valid_id(rid) or rid in role_ids or r.get('_pending_id'):
            add('要確認',f'話者 {rid}','話者IDを確定してください')
        role_ids.add(rid)
        if r.get('kind') not in ('narrator','character'):
            add('要確認',f'話者 {rid}','地の文か登場人物かを確認してください')
    if not project.get('segments'):
        add('エラー','台本','台本項目がありません')
    used = set()
    for s in project.get('segments',[]):
        sid = s.get('id')
        if not _valid_id(sid) or sid in used or s.get('_pending_id'):
            add('要確認',f'項目 {sid}','発話IDを確定してください')
        used.add(sid)
        typ = s.get('segment_type','speech')
        if typ not in ('speech','separator'):
            add('要確認',f'項目 {sid}',f'不明な項目種別: {typ!r}')
            continue
        if typ == 'separator': continue
        if not isinstance(s.get('text'),str) or not s['text'].strip():
            add('エラー',f'発話 {sid}','原文が欠落または空です')
        if s.get('role') not in role_ids:
            add('要確認',f'発話 {sid}',f'未登録の話者ID: {s.get("role")!r}')
    # 入力内の完全性の証明は行わない。検出可能な順序・部分一致のみ検査する。
    source = project.get('source')
    if isinstance(source,str):
        pos = 0
        for s in project.get('segments',[]):
            original = s.get('text')
            if isinstance(original,str) and original:
                idx = source.find(original,pos)
                if idx < 0:
                    add('要確認',f'項目 {s.get("id")}', '原稿全文中に同じ原文を順序どおり見つけられません（部分照合）')
                    break
                pos = idx + len(original)
    return issues

def issues_message(project, preview=12):
    issues = validate_project(project)
    warns = project.get('_load_warnings',[]) if project else []
    lines = [f'要確認・エラー: {len(issues)}件 / 読込時の注意: {len(warns)}件']
    lines += [f'⚠ {level} | {item} | {msg}' for level,item,msg in issues[:preview]]
    lines += [f'ℹ {s}' for s in warns[:max(0,preview-len(issues))]]
    if len(issues)+len(warns)>preview: lines.append(f'ほか {len(issues)+len(warns)-preview} 件')
    if not isinstance(project.get('source') if project else None,str):
        lines.append('原稿全文がないため原文照合は未実施です')
    else:
        lines.append('原文は原稿全文との順序・部分一致のみ照合しました。完全性を保証するものではありません')
    return '\n'.join(lines)

def require_ready(project):
    issues = validate_project(project)
    if issues:
        examples = '\n'.join(f'{item}: {msg}' for _,item,msg in issues[:8])
        raise ValueError(f'生成前に{len(issues)}件の項目を確認してください:\n{examples}')

def export_scenario(project):
    if not project: raise ValueError('シナリオがありません')
    # IDは参照関係の核となるため、仮IDのまま標準形式として出力しない。
    for r in project['roles']:
        if r.get('_pending_id'): raise ValueError(f'話者 {r["id"]}: 仮IDを確定してから保存してください')
    for s in project['segments']:
        if s.get('_pending_id'): raise ValueError(f'発話 {s["id"]}: 仮IDを確定してから保存してください')
    known={r['id'] for r in project['roles']}
    for segment in project['segments']:
        if segment.get('segment_type','speech')=='speech' and segment.get('role') not in known:
            raise ValueError(f'未登録の話者を参照しています: {segment.get("role")}')
    out = copy.deepcopy(project.get('_extras',{}))
    out.update({'schema_version':1,'speakers':{},'segments':[]})
    if project.get('source') is not None: out['source'] = project['source']
    else: out.pop('source',None)
    for r in project['roles']:
        ident = r['id']
        d = copy.deepcopy(r.get('_extras',{}))
        d.update({'name':r['name']})
        if r.get('kind') in ('narrator','character'): d['kind'] = r['kind']
        for src,dst in [('bio','bio'),('caption','voice_caption'),('acting','acting_description')]:
            if r.get(src): d[dst] = r[src]
            else: d.pop(dst,None)
        for attr in ATTR_VALUES:
            if attr in r: d[attr] = r[attr]
            else: d.pop(attr,None)
        out['speakers'][ident] = d
    for s in project['segments']:
        d = copy.deepcopy(s.get('_extras',{}))
        d['id'] = s['id']
        if s.get('role') is not None: d['speaker_id'] = s['role']
        else: d.pop('speaker_id',None)
        if s.get('text') is not None: d['text'] = s['text']
        else: d.pop('text',None)
        d['emotion'] = s.get('emotion','neutral')
        if s.get('acting'): d['acting_description'] = s['acting']
        for k in ('tts_text','pause_after_ms','segment_type','text_format','blank_lines_after'):
            if k in s: d[k] = s[k]
            else: d.pop(k,None)
        out['segments'].append(d)
    return out


def is_narrator(role):
    return role.get('kind') == 'narrator'

# ---------- 決定的ルビ変換（元版 text_conversion.reading と同じ規則・結果） ----------
_KANJI = r'\u3400-\u9fff\U00020000-\U0002ffff々〆〇ヶ'
def reading(body, fmt):
    '''本文を読み上げ用テキストへ変換し、(読み, 警告リスト) を返す。'''
    if fmt not in RUBY_FORMATS: raise ValueError('未対応のルビ形式です')
    if fmt == 'none': return body, []
    warnings, text = [], body
    def sub(pattern, repl, s):
        return re.sub(pattern, repl, s)
    if fmt == 'pixiv':
        text = sub(r'\[\[rb:([^\[\]\n>]+?)\s*>\s*([^\[\]\n]+?)\]\]', lambda m: m[2].strip(), text)
        if '[[' in text or ']]' in text: warnings.append('未対応または不正なpixiv記法があります')
        return text, warnings
    if fmt == 'aozora':
        text = sub(r'※［＃小書き平仮名([ぁ-ゖ])、[0-9０-９]+[-－][0-9０-９]+］', lambda m: m[1], text)
        known = (r'(?:改ページ|改丁|改見開き|ここから[０-９0-9]+字下げ|ここで字下げ終わり|[０-９0-9]+字下げ|'
                 r'「[^「」［］\n]+」に傍点|「[^「」［］\n]+」は[大中小]見出し|[大中小]見出し(?:終わり)?)')
        text = sub(r'［＃([^［］\n]*)］', lambda m: '' if re.fullmatch(known, m[1]) else m[0], text)
    marker = '\ue000'
    while marker in text: marker += '\ue000'
    if fmt == 'kakuyomu': text = text.replace('｜《', marker + 'a')
    if fmt == 'narou':
        text = sub(r'[｜|]([（(])', lambda m: marker + ('b' if m[1] == '(' else 'c'), text)
    if fmt == 'kakuyomu': text = sub(r'《《([^《》\n]+)》》', r'\1', text)
    bars = '｜' if fmt == 'aozora' else '｜|'
    text = sub('[' + bars + r']([^｜|《》\n]+)《([^《》\n]+)》', lambda m: m[2], text)
    text = sub('[' + _KANJI + r']+《([^《》\n]+)》', lambda m: m[1], text)
    if fmt == 'narou':
        text = sub('[' + _KANJI + r']+\(([ぁ-ゖァ-ヺー]+)\)', lambda m: m[1], text)
        text = sub('[' + _KANJI + r']+（([ぁ-ゖァ-ヺー]+)）', lambda m: m[1], text)
    if any(mk in text for mk in ('《', '》', '［＃', '｜', '］')): warnings.append('未対応または不正なルビ・注記があります')
    for suffix, value in (('a', '《'), ('b', '('), ('c', '（')):
        text = text.replace(marker + suffix, value)
    if not text.strip(): warnings.append('変換後に読み上げる本文がありません')
    return text, warnings

# ---------- 音声生成用の自然な発話分割（原文の文字列は厳密に保持） ----------
def split_text(text, limit=140):
    """v1.0.1型の控えめな分割。原文を必ず保持し、短い末尾を孤立させない。

    生成時のみ使用。台本JSONの区切りは変えず、モデル入力の安全のため
    異常に長い部分のみ読点・空白を優先して分割する。句読点がなく長い場合は
    文字数での分割が起き得るため、UIで長文警告・手動調整を推奨する。
    """
    if not isinstance(text,str) or not text.strip():
        raise ValueError('分割する本文が空です')
    if not isinstance(limit,int) or limit<20:
        raise ValueError('分割目安は20文字以上の整数にしてください')
    units = re.findall(r'[^\n。！？!?]*[。！？!?][」』”"）)]*|[^\n。！？!?]*\n+|[^\n。！？!?]+', text)
    if ''.join(units)!=text: raise ValueError('発話分割で本文を保持できませんでした')
    parts,current=[],''
    for unit in units:
        if current and (len(current)+len(unit)>limit or current.endswith('\n')):
            parts.append(current);current=''
        while len(unit)>limit:
            if current:parts.append(current);current=''
            pos=[m.end() for m in re.finditer(r'[、，,；;：:\s]+',unit[:limit])
                 if m.end()>=limit//3 and unit[:m.end()].strip()]
            cut=pos[-1] if pos else limit
            # 数文字だけ余る場合は不自然な切断をせず、長い1単位として扱う。
            if len(unit)-cut < max(12,limit//10) and len(unit)<=int(limit*1.5):
                break
            import regex
            boundaries=[m.end() for m in regex.finditer(r'\X',unit)]
            before=[k for k in boundaries if k<=cut]
            cut=before[-1] if before else boundaries[0]
            if cut<=0 or cut>=len(unit):break
            parts.append(unit[:cut]);unit=unit[cut:]
        current+=unit
    if current:parts.append(current)
    clean,pending=[],''
    for part in parts:
        if not part.strip() and clean:clean[-1]+=part
        elif not part.strip():pending+=part
        else:clean.append(pending+part);pending=''
    if ''.join(clean)!=text or any(not p.strip() for p in clean):
        raise ValueError('空の分割発話があります')
    return clean

print('本体ロジックを読み込みました')


## 3. 合成ロジック（計画・生成・連結）

声の割り当て、生成計画（読み・演技・感情・間の解決）、モデル読み込み、生成と連結です。

In [ ]:
import numpy as np
import soundfile as sf

DEFAULT_PAUSE_MS = 280
SEPARATOR_PAUSE_MS = 1200

def voice_key(project, role):
    return 'shared' if project.get('reading', 'shared') == 'shared' else role['id']

def resolve_pauses(project, global_pause_ms):
    """明示の間 > 区切り・空行 > 共通の間。連続区切りは最大値を適用する。"""
    result, pending_sep, previous = {}, None, None
    for s in project['segments']:
        if s.get('segment_type') == 'separator':
            v = s.get('pause_after_ms', SEPARATOR_PAUSE_MS)
            pending_sep = max(v, pending_sep or 0)
            continue
        if previous is not None and pending_sep is not None:
            if 'pause_after_ms' not in previous:
                result[previous['id']] = max(result[previous['id']], pending_sep)
        # 先頭区切りは前置無音にせず無視し、次の発話間へ持ち越さない。
        pending_sep = None
        if 'pause_after_ms' in s:
            result[s['id']] = s['pause_after_ms']
        elif s.get('blank_lines_after',0) > 0:
            result[s['id']] = max(global_pause_ms, SEPARATOR_PAUSE_MS)
        else:
            result[s['id']] = global_pause_ms
        previous = s
    return result

def build_plan(project, voices, *, use_acting, use_emotion, narrator_emotion, seed, steps, global_pause_ms):
    '''合成リクエストの列を作る。声が未設定なら例外。ルビ未変換が残る発話も例外で止める。'''
    require_ready(project)
    roles = {r['id']: r for r in project['roles']}
    default_fmt = (project.get('textOptions') or {}).get('format', 'none')
    pauses = resolve_pauses(project, global_pause_ms)
    missing = []
    for r in project['roles']:
        if project.get('reading', 'shared') == 'shared':
            if 'shared' not in voices: missing.append('共用の声')
            break
        if voice_key(project, r) not in voices: missing.append(r['name'])
    if missing: raise ValueError('参照音声が未設定です: ' + '、'.join(missing) + '（配役・声タブで採用または持ち込みをしてください）')
    plan, blocked = [], []
    STATE['plan_warnings'] = []
    for s in project['segments']:
        if s.get('segment_type') == 'separator': continue
        role = roles[s['role']]
        fmt = s.get('text_format', default_fmt)
        if isinstance(s.get('tts_text'), str) and s['tts_text'].strip():
            spoken, warns = s['tts_text'], []
        else:
            spoken, warns = reading(s['text'], fmt)
        if warns: blocked.append(f"{s['id']}: {' / '.join(warns)}")
        pieces = split_text(spoken, 140)
        for piece in pieces:
            if len(piece.strip()) < 10:
                STATE['plan_warnings'].append(f"{s['id']}: {len(piece.strip())}文字「{piece.strip()}」— 極端に短い発話は合成が不安定な場合があります（必要な場合だけ台本を調整してください）")
        for i, piece in enumerate(pieces):
            text = piece
            if i == 0 and use_emotion and role.get('kind') in ('narrator','character') and (narrator_emotion or not is_narrator(role)):
                text = EMOTIONS[s['emotion']] + text
            # クローン時は元版と同じく共通の baseCaption を土台にする（人物の caption は声質＝試作用。声の同一性は参照WAVが担う）
            caption = project.get('baseCaption', '')
            if use_acting:
                caption = '\n'.join(p for p in [caption.rstrip(), role.get('acting', '').strip(), s.get('acting', '').strip()] if p)
            ref = voices[voice_key(project, role)]
            plan.append({'unit_id': s['id'] if len(pieces) == 1 else f"{s['id']}#{i+1}", 'segment_id': s['id'],
                         'text': text, 'caption': caption or None, 'ref_wav': str(ref),
                         'seed': seed, 'num_steps': steps,
                         'pause_after_ms': pauses.get(s['id'], global_pause_ms) if i == len(pieces) - 1 else 0})
    if blocked: raise ValueError('読み変換で確認が必要な発話があります。tts_textで読みを指定するか、記法を直してください:\n' + '\n'.join(blocked))
    if not plan: raise ValueError('読み上げる本文がありません')
    return plan

def request_key(unit, model_id):
    h = hashlib.sha256()
    h.update(Path(unit['ref_wav']).read_bytes())
    ref_hash = h.hexdigest()
    return hashlib.sha256(json.dumps({'model': model_id, 'model_revision':MODEL_REVISION, 'codec':CODEC_ID, 'codec_revision':CODEC_REVISION, 'upstream':globals().get('IRODORI_RESOLVED_COMMIT',globals().get('IRODORI_COMMIT','unknown')), 'ref': ref_hash, **{k: unit[k] for k in ('text', 'caption', 'seed', 'num_steps')}}, ensure_ascii=False, sort_keys=True).encode()).hexdigest()

def cache_path(unit_id, cache_key):
    """区切り文字・Unicode・ID衝突・パストラバーサルを避けたファイル名。"""
    digest = hashlib.sha256((unit_id + ':' + cache_key).encode('utf-8')).hexdigest()
    return SEG_DIR / f'unit_{digest}.wav'

def _load_runtime_unlocked(model_id=MODEL_ID):
    if _RUNTIME['instance'] is not None and _RUNTIME['model'] == model_id:
        return _RUNTIME['instance']
    import sys
    sys.path.insert(0, str(IRODORI_DIR))
    from huggingface_hub import hf_hub_download
    from irodori_tts.inference_runtime import InferenceRuntime, RuntimeKey
    ckpt = hf_hub_download(model_id, 'model.safetensors', revision=MODEL_REVISION)
    codec = hf_hub_download(CODEC_ID, 'weights.pth', revision=CODEC_REVISION)
    runtime = InferenceRuntime.from_key(RuntimeKey(checkpoint=ckpt, codec_repo=codec,
                                                   model_device='cuda', codec_device='cuda', model_precision='fp32', codec_precision='fp32'))
    _RUNTIME['instance'], _RUNTIME['model'] = runtime, model_id
    return runtime

import threading
_RUNTIME_LOCK=threading.Lock()
def load_runtime(model_id=MODEL_ID):
    with _RUNTIME_LOCK:
        return _load_runtime_unlocked(model_id)

def synthesize(unit, runtime):
    from irodori_tts.inference_runtime import SamplingRequest
    result = runtime.synthesize(SamplingRequest(text=unit['text'], caption=unit['caption'], ref_wav=unit['ref_wav'],
                                                max_ref_seconds=30, no_ref=False, num_steps=unit['num_steps'], seed=unit['seed']))
    audio = result.audios[0].detach().cpu().float().view(-1).numpy()
    if not audio.size or not np.isfinite(audio).all(): raise RuntimeError('空または異常な音声が生成されました')
    return audio, int(result.sample_rate)

def generate_all(plan, model_id=MODEL_ID, progress=None):
    runtime = load_runtime(model_id)
    files = []
    for n, unit in enumerate(plan, 1):
        key = request_key(unit, model_id)
        wav = cache_path(unit['unit_id'], key)
        meta = wav.with_suffix('.json')
        if wav.exists() and meta.exists() and json.loads(meta.read_text(encoding='utf-8')).get('key') == key:
            if progress: progress(f'[{n}/{len(plan)}] {unit["unit_id"]} は生成済みのため再利用')
        else:
            if progress: progress(f'[{n}/{len(plan)}] {unit["unit_id"]} を生成中…')
            audio, sr = synthesize(unit, runtime)
            sf.write(str(wav), audio, sr, subtype='PCM_16')
            meta.write_text(json.dumps({'key':key, 'unit_id':unit['unit_id'], 'segment_id':unit['segment_id']},ensure_ascii=False), encoding='utf-8')
        files.append((unit, wav))
    return files

def concat(files):
    if not files: raise ValueError('連結する音声がありません')
    chunks, sr_all = [], None
    for unit, wav in files:
        audio, sr = sf.read(str(wav), dtype='float32')
        if sr_all is None: sr_all = sr
        if sr != sr_all: raise ValueError('サンプルレートが混在しています')
        chunks.append(audio)
        pause = unit.get('pause_after_ms', 0)
        if pause and (unit, wav) != files[-1]:
            chunks.append(np.zeros(int(sr * pause / 1000), dtype='float32'))
    out = session_out_path('narration.wav')
    sf.write(str(out), np.concatenate(chunks), sr_all, subtype='PCM_16')
    return out

print('合成ロジックを読み込みました')

## 4. 画面（Web UI）

実行すると Gradio の画面が起動し、リンクが表示されます。画面の流れに沿って操作してください。
モデルの読み込みは、試作または本編の生成時に自動で行われます（初回は数分かかります）。

In [ ]:
import gradio as gr
import pandas as pd
import copy, json, random, shutil
from pathlib import Path

SAMPLE_TEXT_DEFAULT = '今日はとても良い天気ですね。こんな日は、少し遠くまで散歩に出かけたくなります。途中で見つけた小さな店で、温かい飲み物を買って、のんびり帰りましょう。'

# 履歴（Undoスタック）
MAX_UNDO = 10

def push_undo():
    if STATE.get('project'):
        stack = STATE.setdefault('undo', [])
        stack.append(copy.deepcopy({k:STATE.get(k) for k in ('project','voices','sample')}))
        del stack[:-MAX_UNDO]
        STATE['epoch'] = STATE.get('epoch',0) + 1
        STATE.pop('length_approval',None)


def pop_undo():
    stack = STATE.setdefault('undo', [])
    if not stack: return '元に戻す履歴がありません'
    STATE.update(stack.pop())
    STATE['epoch'] = STATE.get('epoch',0) + 1
    STATE['last_files'] = []
    STATE.pop('length_approval',None)
    return '1つ前の状態に戻しました'


def project_summary():
    p = STATE.get('project')
    if not p: return '【シナリオ未読込】左上の読込パネルからJSONを指定してください'
    voices = STATE.get('voices', {})
    lines = [f"📖 {p.get('title','無題')} | 話者: {len(p['roles'])}人 | 発話: {len(p['segments'])}件 | 方式: {'共通' if p.get('reading','shared')=='shared' else '話者別'}"]
    for r in p['roles']:
        key = voice_key(p, r)
        status = '✅ 設定済み' if key in voices else '❌ 未設定'
        lines.append(f"  • {r['name']} ({r['id']}): {status}")
    return '\n'.join(lines)

def get_role_choices():
    p = STATE.get('project')
    if not p: return []
    return [(f"{r['name']}［{r['id']}］", r['id']) for r in p['roles']]

def find_role_by_name(name):
    p = STATE.get('project')
    if not p: return None
    return next((r for r in p['roles'] if r['id'] == name), None)

def current_role_wav(role_name):
    p = STATE.get('project')
    if not p or not role_name: return None
    r = find_role_by_name(role_name)
    if not r: return None
    key = voice_key(p, r)
    return STATE.get('voices', {}).get(key, None)

def load_scenario_action(file, paste, reading_choice):
    try:
        raw_text = None
        if file is not None:
            raw_text = Path(file).read_text(encoding='utf-8-sig')
        elif paste and paste.strip():
            raw_text = paste
        if raw_text is None:
            return 'JSONファイルを選ぶか、JSONを貼り付けてください', project_summary(), gr.update(choices=[]), None, *render_table(0, 50), issues_message(STATE.get('project'))
        project = normalize_scenario(json.loads(raw_text))
        if reading_choice in ('shared', 'cast'):
            project['reading'] = reading_choice
        # 読み込みに成功したときだけ切り替え、前の作品の声・試作・キャッシュ記録を引き継がない
        STATE['epoch'] = STATE.get('epoch',0) + 1
        STATE.pop('length_approval',None)
        STATE['project'] = project
        STATE['voices'] = {}
        STATE['sample'] = None
        STATE['last_files'] = []
        STATE.setdefault('undo',[]).clear()
        choices = get_role_choices()
        first = choices[0][1] if choices else None
        tb_df, tb_stat, tb_page = render_table(0, 50)
        msg = f'読み込み完了: 台本項目{len(project["segments"])}件 / 話者{len(project["roles"])}人（確認事項は右の欄をご覧ください）'
        return msg, project_summary(), gr.update(choices=choices, value=first), None, tb_df, tb_stat, tb_page, issues_message(project)
    except Exception as e:
        choices = get_role_choices()
        first = choices[0][1] if choices else None
        return f'読み込み失敗: {e}', project_summary(), gr.update(choices=choices, value=first), current_role_wav(first), *render_table(0, 50), issues_message(STATE.get('project'))

def sample_caption_prefill(role_name):
    r = find_role_by_name(role_name)
    if not r: return ''
    return r.get('caption','')

def on_role_change(role_id):
    wav = current_role_wav(role_id)
    cap = sample_caption_prefill(role_id)
    r = find_role_by_name(role_id)
    if not r: return wav,cap,'','',None,'',''
    return wav,cap,r.get('id',''),r.get('name',''),r.get('kind'),r.get('bio',''),r.get('acting','')

def save_role_action(old_id, new_id, name, kind, bio, voice_caption, acting):
    p = STATE.get('project')
    if not p: return 'シナリオを読み込んでください',project_summary(),gr.update(),None
    r = find_role_by_name(old_id)
    if not r: return '話者を選択してください',project_summary(),gr.update(),None
    if not isinstance(new_id,str) or not new_id.strip():
        return '話者IDは空にできません',project_summary(),gr.update(),None
    if new_id != old_id and any(q['id']==new_id for q in p['roles']):
        return '同じ話者IDがすでに存在します',project_summary(),gr.update(),None
    if not isinstance(name,str) or not name.strip():
        return '話者の表示名を入力してください',project_summary(),gr.update(),None
    if kind not in ('narrator','character'):
        return '「地の文」か「登場人物」を選択してください',project_summary(),gr.update(),None
    push_undo()
    # 明示的なID変更に限り、参照発話IDと既存の声の割り当てを追随させる。
    old_voice_key = voice_key(p,r)
    r.update(id=new_id,name=name,kind=kind,bio=bio or '',caption=voice_caption or '',acting=acting or '',_pending_id=False)
    if new_id != old_id:
        for s in p['segments']:
            if s.get('role') == old_id: s['role'] = new_id
        new_voice_key = voice_key(p,r)
        if old_voice_key in STATE.get('voices',{}):
            STATE['voices'][new_voice_key] = STATE['voices'].pop(old_voice_key)
        sample = STATE.get('sample')
        if sample and sample.get('role_name') == old_id: sample['role_name'] = new_id
    STATE['last_files'] = []
    return '話者の設定を保存しました',project_summary(),gr.update(choices=get_role_choices(),value=new_id),current_role_wav(new_id)

def gen_sample_action(role_name, caption, text, seed, steps):
    try:
        if STATE.get('project') is None: return '先にシナリオを読み込んでください', None
        if not text.strip() or len(text) > 200: return '試作文は1〜200文字にしてください', None
        runtime = load_runtime()
        from irodori_tts.inference_runtime import SamplingRequest
        result = runtime.synthesize(SamplingRequest(text=text, caption=caption or None, no_ref=True, num_steps=int(steps), seed=int(seed), max_seconds=30.0))
        audio = result.audios[0].detach().cpu().float().view(-1).numpy()
        safe_name = hashlib.sha256(role_name.encode('utf-8')).hexdigest()[:20]
        import uuid
        path = session_out_path(f'sample_{safe_name}_{uuid.uuid4().hex}.wav')
        sf.write(str(path), audio, int(result.sample_rate), subtype='PCM_16')
        STATE['sample'] = {'path': str(path), 'role_name': role_name}
        return f'【{role_name}】の試作音声を生成しました。試聴して問題なければ採用してください。', str(path)
    except Exception as e:
        return f'試作エラー: {e}', None

def adopt_sample_action(role_name):
    p, s = STATE.get('project'), STATE.get('sample')
    if not p or not s: return '試作音声がありません', project_summary(), None
    if s.get('role_name') not in (None, role_name):
        return f"その試作は【{s.get('role_name')}】用に作ったものです。【{role_name}】へは割り当てません。話者を選び直すか、その話者で試作を作り直してください", project_summary(), None
    r = find_role_by_name(role_name)
    if not r: return '話者が未選択です', project_summary(), None
    key = voice_key(p, r)
    import uuid
    push_undo()
    dest = session_out_path(f'voice_{uuid.uuid4().hex}.wav')
    shutil.copyfile(s['path'], dest)
    STATE.setdefault('voices', {})[key] = str(dest)
    STATE['last_files'] = []
    return f'【{role_name}】に試作音声を割り当てました', project_summary(), str(dest)

def adopt_upload_action(role_name, audio_path):
    p = STATE.get('project')
    if not p or not audio_path: return '音声ファイルが指定されていません', project_summary(), None
    r = find_role_by_name(role_name)
    if not r: return '話者が未選択です', project_summary(), None
    key = voice_key(p, r)
    import uuid
    push_undo()
    dest = session_out_path(f'voice_{uuid.uuid4().hex}.wav')
    shutil.copyfile(audio_path, dest)
    STATE.setdefault('voices', {})[key] = str(dest)
    STATE['last_files'] = []
    return f'【{role_name}】に持ち込み音声を割り当てました', project_summary(), str(dest)

def download_refs_zip():
    voices = STATE.get('voices', {})
    if not voices: return None
    import zipfile
    zpath = session_out_path('reference_voices.zip')
    with zipfile.ZipFile(zpath, 'w') as z:
        manifest = {}
        for key, path in voices.items():
            if Path(path).exists():
                z.write(path, arcname=Path(path).name)
                manifest[key] = Path(path).name
        z.writestr('manifest.json',json.dumps(manifest,ensure_ascii=False,indent=2))
    return str(zpath)

def download_segments_zip():
    import zipfile
    files = STATE.get('last_files') or []
    if not files:
        raise gr.Error('最初に「朗読音声を生成」を実行してください。直近の全体生成で使用した音声のみ保存します')
    zpath = session_out_path('segments_all.zip')
    with zipfile.ZipFile(zpath, 'w') as z:
        manifest=[]
        for unit,w in files:
            meta = w.with_suffix('.json')
            if not w.exists() or not meta.exists() or json.loads(meta.read_text(encoding='utf-8')).get('key') != request_key(unit, MODEL_ID):
                raise gr.Error(f'個別音声のキャッシュが一致しません: {unit["unit_id"]}。全体生成をやり直してください')
            z.write(w, arcname=w.name)
            z.write(meta, arcname=meta.name)
            manifest.append({'file':w.name,'unit_id':unit['unit_id'],'segment_id':unit['segment_id']})
        z.writestr('manifest.json',json.dumps(manifest,ensure_ascii=False,indent=2))
    return str(zpath)



# --- 台本エディタ（DataFrameベース + アクションバー） ---
EMOTION_JP = {'neutral':'平静','joy':'喜び','anger':'怒り','sad':'悲しみ','kind':'優しさ',
    'tease':'からかい','anxiety':'不安','tension':'緊張','surprise':'驚き',
    'tired':'疲れ','whisper':'ささやき','giggle':'忍び笑い','fear':'恐怖',
    'sleepy':'眠気','resolve':'決意・強調'}
def display_emotion(value):
    code=value if value in EMOTIONS else 'neutral'
    icon=EMOTIONS[code] or '―'
    return f'{icon} {EMOTION_JP[code]} ({code})'

def parse_emotion(value):
    value=_field_value(value).strip()
    if value in EMOTIONS:return value
    m=re.search(r'\(([a-z_]+)\)\s*$',value)
    if m and m.group(1) in EMOTIONS:return m.group(1)
    for code in EMOTIONS:
        if value==display_emotion(code):return code
    raise ValueError(f'感情の指定 {value!r} は未対応です（joy等のIDも入力できます）')

EDIT_COLUMNS = ['#', '発話ID', '話者ID', '原文', '読み上げ文', '発話ごとの演技指示', '感情（絵文字・名前）', '発話後の間(ms)', '長さの確認']
TABLE_WIDTHS = ['5%', '8%', '9%', '26%', '24%', '12%', '12%', '8%', '12%']

def speech_length(s):
    text=s.get('tts_text') if isinstance(s.get('tts_text'),str) and s['tts_text'].strip() else s.get('text')
    if not isinstance(text,str):return None
    if text == s.get('text'):
        try:
            txt,warnings=reading(text,s.get('text_format','none'))
            if not warnings:text=txt
        except (ValueError,TypeError):pass
    return len(text.strip())

def inherit_review(target,other):
    meta=other.get('_extras',{}).get('koebuta_edit',{})
    if not isinstance(meta,dict):return
    own=target.setdefault('_extras',{}).setdefault('koebuta_edit',{})
    if not isinstance(own,dict):return
    for key in ('forced','fallback'):
        if meta.get(key):own[key]=True

def mark_review(segment, action, fallback=False):
    extra=segment.setdefault('_extras',{})
    old=extra.get('koebuta_edit',{})
    actions=[x for x in old.get('actions',[]) if isinstance(x,str)] if isinstance(old,dict) and isinstance(old.get('actions'),list) else []
    if action not in actions: actions.append(action)
    extra['koebuta_edit']={'actions':actions,'needs_review':True,
                          'fallback':bool(fallback or (isinstance(old,dict) and old.get('fallback'))),
                          'forced':bool(isinstance(old,dict) and old.get('forced'))}

def review_status(segment):
    meta=segment.get('_extras',{}).get('koebuta_edit',{})
    if not isinstance(meta,dict) or not isinstance(meta.get('actions'),list): return ''
    actions=[x for x in meta['actions'] if isinstance(x,str)]
    if not actions:return ''
    names={'split':'✂ 自動分割','merge':'🔗 自動結合','manual_split':'✂ 分割','manual_merge':'🔗 結合'}
    text='／'.join(names.get(x,x) for x in actions)
    text+='・要確認' if meta.get('needs_review') else '・確認済み'
    if meta.get('fallback'): text+='（自然な区切りなし）'
    if meta.get('forced'): text+=' ⚠ 強制処理・原文と読みの対応を確認'
    return text

def project_stamp():
    data={'project':STATE.get('project'),'voices':STATE.get('voices'), 'epoch':STATE.get('epoch',0)}
    return hashlib.sha256(json.dumps(data,ensure_ascii=False,sort_keys=True).encode()).hexdigest()

def _length_status(s, low=None, high=None):
    if s.get('segment_type')=='separator':return '区切り'
    low=int(low if low is not None else STATE.get('length_min',12))
    high=int(high if high is not None else STATE.get('length_max',140))
    size=speech_length(s)
    if size is None:return '🟥 原文なし'
    if size < low:return f'🟥 短文 {size}字'
    if size > high:return f'🟥 長文 {size}字'
    return f'✓ {size}字'


def length_status(s,low=None,high=None):
    return ' | '.join(x for x in (_length_status(s,low,high),review_status(s)) if x)

def render_table(start=0, count=50):
    p = STATE.get('project')
    if not p or not p.get('segments'):
        df = pd.DataFrame(columns=EDIT_COLUMNS)
        STATE.pop('table_view',None)
        return df, 'シナリオ未読込', '0 / 0 件'
    total = len(p['segments'])
    start = max(0, min(int(start or 0), max(0, total - 1)))
    end = min(total, start + int(count or 50))
    role_map = {r['id']: r['name'] for r in p['roles']}
    
    rows = []
    for idx, s in enumerate(p['segments'][start:end], start=start):
        rows.append({
            '#': idx,
            '発話ID': s.get('id',''),
            '話者ID': s.get('role') or '',
            '原文': s.get('text') if isinstance(s.get('text'),str) else '',
            '読み上げ文': s.get('tts_text', ''),
            '発話ごとの演技指示': s.get('acting', ''),
            '感情（絵文字・名前）': display_emotion(s.get('emotion', 'neutral')),
            '発話後の間(ms)': s.get('pause_after_ms', ''),
            '長さの確認': length_status(s)
        })
    df = pd.DataFrame(rows, columns=EDIT_COLUMNS)
    STATE['table_view'] = {'start':start,'count':int(count or 50),'stamp':project_stamp(),
                           'ids':[x['id'] for x in p['segments'][start:end]],'data':df.copy(deep=True)}
    info_msg = f'{start}〜{end - 1} 行目を表示中'
    page_msg = f'全 {total} 件中 ({start + 1}〜{end})'
    return df, info_msg, page_msg

def _field_value(value):
    if value is None or (not isinstance(value,str) and pd.isna(value)):
        return ''
    return str(value)

def _pause_value(v):
    if v is None or (not isinstance(v,str) and pd.isna(v)) or (isinstance(v,str) and not v.strip()):
        return None
    if isinstance(v,bool): raise ValueError('間には0以上の整数を指定してください')
    try:
        number = float(v)
        if not math.isfinite(number) or number < 0 or not number.is_integer(): raise ValueError()
        return int(number)
    except (ValueError,TypeError,OverflowError):
        raise ValueError('間には0以上の整数（ミリ秒）を指定してください')

def save_table_edits(df, start, count):
    p = STATE.get('project')
    if not p: return 'シナリオ未読込です', project_summary(), *render_table(start, count)
    try:
        view=STATE.get('table_view')
        if not view or view['stamp']!=project_stamp():
            raise ValueError('表示後に台本が変わりました。入力内容を控え、表を更新してください')
        # 入力欄を変えただけでは保存先を変えない。実際に表示したページに保存する。
        start,count=view['start'],view['count']
        end = min(len(p['segments']),start+count)
        if [x['id'] for x in p['segments'][start:end]]!=view['ids']:
            raise ValueError('表示元の発話が変わっています。表を更新してください')
        original = p['segments'][start:end]
        if len(df) != len(original):
            raise ValueError('表示中の行数が変わっています。削除・結合は専用ボタンから行ってください')
        if [_field_value(x) for x in df['#']] != [str(x) for x in range(start,end)]:
            raise ValueError('行番号または行順が変わっています。表を更新してください')
        new_slice = []
        for (_,row),old in zip(df.iterrows(),original):
            seg = copy.deepcopy(old)
            sid = _field_value(row['発話ID'])
            if not sid.strip(): raise ValueError('発話IDを空にできません')
            if sid != old['id']:
                seg['_pending_id'] = False
            seg['id'] = sid
            role_id = _field_value(row['話者ID'])
            seg['role'] = role_id or None
            if seg.get('segment_type','speech')=='speech' and role_id not in {r['id'] for r in p['roles']}:
                raise ValueError(f'未登録の話者IDです: {role_id}')
            # 原文は空白も含め、そのまま保持する
            new_text = _field_value(row['原文'])
            seg['text'] = old.get('text') if not isinstance(old.get('text'),str) and new_text == '' else new_text
            for label, dest in [('読み上げ文','tts_text'),('発話ごとの演技指示','acting')]:
                value = _field_value(row[label])
                if value.strip(): seg[dest] = value
                else: seg.pop(dest,None)
            emo = parse_emotion(row['感情（絵文字・名前）'])
            seg['emotion'] = emo
            pause = _pause_value(row['発話後の間(ms)'])
            if pause is None: seg.pop('pause_after_ms',None)
            else: seg['pause_after_ms'] = pause
            new_slice.append(seg)
        candidate = copy.deepcopy(p)
        candidate['segments'] = p['segments'][:start] + new_slice + p['segments'][end:]
        ids = [s['id'] for s in candidate['segments']]
        if len(ids) != len(set(ids)):
            raise ValueError('発話IDが別ページを含む全体で重複しています')
        if candidate==p:
            data,_,pages=render_table(start,count)
            return '表の編集を反映しました（変更なし）',project_summary(),data,'変更なし',pages
        push_undo()
        STATE['project'] = candidate
        STATE['last_files'] = []
        status = f'表の編集を反映しました（台本項目{len(ids)}件）。音声生成前の確認事項もご覧ください'
        data,_,pages = render_table(start,count)
        return status,project_summary(),data,status,pages
    except Exception as e:
        # エラー時は入力表をそのまま返し、修正中の内容と表示元を保護する。
        return f'保存できません: {e}',project_summary(),df,f'エラー: {e}','入力を保持しています'



def undo_action(start, count):
    msg = pop_undo()
    df_out, _, page_msg = render_table(start, count)
    return msg, project_summary(), df_out, msg, page_msg

# --- 選択した発話の結合・分割 ---
def _unique_part_id(project, prefix):
    used={s['id'] for s in project['segments']}
    i=2
    while f'{prefix}_part{i}' in used:i+=1
    return f'{prefix}_part{i}'

def merge_row_with_next(target_idx, start, count):
    p=STATE.get('project')
    if not p or not p.get('segments'):
        return 'シナリオがありません', *render_table(start,count)
    try:
        idx=int(target_idx)
        if idx<0 or idx+1>=len(p['segments']):
            raise ValueError('最後の行・範囲外の行は次と結合できません')
        top,bot=p['segments'][idx:idx+2]
        if any(s.get('segment_type','speech')!='speech' for s in (top,bot)):
            raise ValueError('区切り行は結合できません')
        if top.get('role')!=bot.get('role'):
            raise ValueError('話者が異なるため結合しません。話者IDを確認してください')
        if not all(isinstance(s.get('text'),str) and s['text'] for s in (top,bot)):
            raise ValueError('原文が欠落しているため結合できません')
        source=p.get('source')
        gap=''
        if isinstance(source,str):
            gap=_source_gap_for_pair(p,idx)
            if gap is None:
                raise ValueError('原稿全文と発話順序が一致しません。原文を確認してください')
            if gap.strip():
                raise ValueError('原文の間に別の文章があります。欠落を防ぐため結合しません')
        new=copy.deepcopy(top)
        new['text']=top['text']+gap+bot['text']
        inherit_review(new,bot)
        mark_review(new,'manual_merge')
        notes=[]
        if top.get('emotion','neutral')!=bot.get('emotion','neutral'):
            notes.append('感情は前の発話を採用しました')
        if (top.get('acting') or '')!=(bot.get('acting') or ''):
            notes.append('演技指示は前の発話を優先しました' if top.get('acting') else '演技指示は後の発話から引き継ぎました')
        if top.get('pause_after_ms') is not None or top.get('blank_lines_after',0):
            notes.append('発話間にあった指定の間・空行数は一本化により失われました')
        f1=top.get('text_format','none');f2=bot.get('text_format','none')
        # 片方だけがルビ形式を使用しているなら、その形式を引き継ぐ。
        # 異なるルビ形式が混在するときは、読みを各々展開してtts_textに固定する。
        format_mixed=f1!=f2 and f1!='none' and f2!='none'
        def spoken(s):
            if isinstance(s.get('tts_text'),str) and s['tts_text'].strip():return s['tts_text']
            converted,warnings=reading(s['text'],s.get('text_format','none'))
            if warnings:raise ValueError(f'{s["id"]}: 読み変換が不確実です。読み上げ文を先に調整してください')
            return converted
        # 結合境界をまたいでルビの親文字が広がっても、各行の読みを保持する。
        expected_spoken=spoken(top)+gap+spoken(bot)
        merged_spoken,merged_warnings=reading(new['text'],f2 if f1=='none' else f1) if not format_mixed else ('',[])
        if top.get('tts_text') or bot.get('tts_text') or format_mixed or merged_warnings or merged_spoken!=expected_spoken:
            new['tts_text']=expected_spoken
        if format_mixed:
            new.pop('text_format',None)
            notes.append('異なるルビ形式は読み上げ文に変換して統合しました')
        elif f1=='none' and f2!='none':
            new['text_format']=f2
        if not new.get('acting'):
            new['acting']=bot.get('acting','')
        for field in ('pause_after_ms','blank_lines_after'):
            if field in bot:new[field]=bot[field]
            else:new.pop(field,None)
        # 手動操作なので、統合で上書きされる後半の設定を結果として必ず知らせる。
        push_undo()
        p['segments'][idx:idx+2]=[new]
        STATE['last_files']=[]
        msg=f'行{idx}と{idx+1}を結合しました。'+(' ⚠️ '+'／'.join(notes) if notes else '')
        df,_,page=render_table(start,count)
        return msg,df,msg,page
    except Exception as e:
        df,_,page=render_table(start,count)
        return f'結合できません: {e}',df,f'エラー: {e}',page

def _split_boundaries(text):
    import regex
    boundaries={m.end() for m in regex.finditer(r'\X',text)}
    boundaries.discard(len(text))
    patterns=[r'[｜|][^｜|《》\n]+《[^《》\n]+》',
              '['+_KANJI+r']+《[^《》\n]+》',
              '['+_KANJI+r']+[（(][ぁ-ゖァ-ヺー]+[）)]',
              r'\[\[rb:[^\n]*?\]\]', r'《《[^\n]*?》》',r'※?［＃[^\n]*?］',
              r'[…―—]{2,}']
    for pattern in patterns:
        for m in re.finditer(pattern,text):
            boundaries.difference_update(range(m.start()+1,m.end()))
    # 不完全なルビ・注記の中も避ける。
    opened=None
    for match in re.finditer(r'《|》|［＃|］',text):
        if match.group() in ('《','［＃'):opened=match.start() if opened is None else opened
        elif opened is not None:
            boundaries.difference_update(range(opened+1,match.end()));opened=None
    if opened is not None:boundaries.difference_update(range(opened+1,len(text)))
    return boundaries

def _ruby_safe_offset(text,offset):
    return offset in _split_boundaries(text)


def _split_choice(text,target=None,allow_unsafe=False):
    target=float(target if target is not None else len(text)/2)
    target=max(1,min(target,len(text)-1))
    # 候補範囲を目標の前後へ限定し、冒頭の句点だけで極端な短文を作らない。
    floor=max(1,int(target*0.50));ceiling=min(len(text)-1,int(target*1.30))
    closing='」』”"）)】〕〉》'
    candidates=[]
    if allow_unsafe:
        import regex
        boundaries={m.end() for m in regex.finditer(r'\X',text)}-{len(text)}
    else:boundaries=_split_boundaries(text)
    patterns=[(0,r'\n[ \t\r]*\n+'),(1,r'[。！？!?]+['+re.escape(closing)+r']*'),
              (2,r'\n+'),(3,r'[、，,；;：:]+['+re.escape(closing)+r']*'),
              (4,r'[…―—]{2,}['+re.escape(closing)+r']*'),
              (5,r'[」』）)】]'),(6,r'\s+')]
    for rank,pattern in patterns:
        for m in re.finditer(pattern,text):
            k=m.end()
            if floor<=k<=ceiling and k in boundaries and text[:k].strip() and text[k:].strip():
                candidates.append((rank,abs(k-target),k))
    if candidates: return min(candidates)[2],False
    # 文法的な自然さは保証しない。フォールバックは表示して人に確認してもらう。
    safe=[k for k in sorted(boundaries) if text[:k].strip() and text[k:].strip()]
    if not safe:return None,True
    return min(safe,key=lambda k:abs(k-target)),True

def _natural_split_offset(text):
    return _split_choice(text)[0]


def split_single_row(target_idx, split_at, start, count):
    p=STATE.get('project')
    if not p or not p.get('segments'):
        return 'シナリオがありません',*render_table(start,count)
    try:
        idx=int(target_idx)
        if idx<0 or idx>=len(p['segments']):raise ValueError('行番号が範囲外です')
        original=p['segments'][idx]
        if original.get('segment_type','speech')!='speech':raise ValueError('区切り行は分割しません')
        text=original.get('text')
        if not isinstance(text,str) or len(text)<2:raise ValueError('分割できる原文がありません')
        specified=split_at is not None and str(split_at).strip()!=''
        k=int(split_at) if specified else _natural_split_offset(text)
        if k is None:
            raise ValueError('自然な分割位置が見つかりません。先頭からの文字数を指定してください')
        if not _ruby_safe_offset(text,k):
            raise ValueError(f'位置{k}は文の端またはルビ・結合文字の途中です。別の位置を指定してください')
        if not text[:k].strip() or not text[k:].strip():
            raise ValueError('分割後の両方に読み上げる文章が必要です')
        has_tts=isinstance(original.get('tts_text'),str) and bool(original['tts_text'].strip())
        if has_tts and original['tts_text']!=text:
            raise ValueError('原文と異なる読み上げ文があるため自動で対応箇所を割り出せません。読み上げ文を分割前に個別に調整してください')
        if not has_tts:
            fmt=original.get('text_format','none')
            whole,w=reading(text,fmt);left,lw=reading(text[:k],fmt);right,rw=reading(text[k:],fmt)
            if w or lw or rw or left+right!=whole:
                raise ValueError('分割前後で読みが一致しません。ルビを含まない位置を指定してください')
        first=copy.deepcopy(original);second=copy.deepcopy(original)
        first['text']=text[:k];second['text']=text[k:]
        second['id']=_unique_part_id(p,original['id'])
        fallback=not specified and _split_choice(text)[1]
        mark_review(first,'manual_split',fallback);mark_review(second,'manual_split',fallback)
        if has_tts:
            first['tts_text']=original['tts_text'][:k]
            second['tts_text']=original['tts_text'][k:]
        # 終端の後の間・空行数を後半に移動。他の感情や演技は両方へ継承する。
        for key in ('pause_after_ms','blank_lines_after'):first.pop(key,None)
        proposed=copy.deepcopy(p)
        proposed['segments'][idx:idx+1]=[first,second]
        if len(validate_project(proposed))>len(validate_project(p)):
            raise ValueError('分割後に確認事項が増えるため中止しました')
        push_undo()
        p['segments'][idx:idx+1]=[first,second]
        STATE['last_files']=[]
        msg=f'行{idx}を位置{k}で2つに分割しました。後半のIDは {second["id"]}。両方へ感情・演技指示を引き継ぎました'
        df,_,page=render_table(start,count)
        return msg,df,msg,page
    except Exception as e:
        df,_,page=render_table(start,count)
        return f'分割できません: {e}',df,f'エラー: {e}',page


# --- 発話長の警告と、安全側の分割・結合（適用前にプレビュー） ---
def _source_gap_for_pair(project, index):
    """重複する文章にも対応し、作品先頭から順に位置を確認する。"""
    source=project.get('source')
    if not isinstance(source,str):return None
    cursor=0
    segs=project['segments']
    for pos,s in enumerate(segs[:index+2]):
        text=s.get('text')
        if s.get('segment_type')=='separator' and not text:continue
        if not isinstance(text,str) or not text:return None
        found=source.find(text,cursor)
        if found<0:return None
        if pos==index: end=found+len(text)
        if pos==index+1:return source[end:found]
        cursor=found+len(text)
    return None

def _safe_auto_merge(project,index,low,high):
    segs=project['segments']
    if index>=len(segs)-1:return False,'末尾'
    top,bot=segs[index:index+2]
    if any(s.get('segment_type','speech')!='speech' for s in (top,bot)):
        return False,'区切り行を含む'
    if top.get('role')!=bot.get('role') or top.get('emotion')!=bot.get('emotion'):
        return False,'話者または感情が異なる'
    if top.get('acting','')!=bot.get('acting','') or top.get('text_format','none')!=bot.get('text_format','none'):
        return False,'演技指示またはルビ形式が異なる'
    if top.get('text_format','none')!='none' or any(s.get('tts_text') for s in (top,bot)):
        return False,'ルビ・読み上げ文は自動統合しない'
    if top.get('pause_after_ms') is not None or top.get('blank_lines_after',0):
        return False,'明示的な間または空行がある'
    gap=_source_gap_for_pair(project,index) if isinstance(project.get('source'),str) else ''
    if gap is None:return False,'原稿全文で隣接を確認できない'
    if '\r' in gap or '\n' in gap or gap.strip():
        return False,'改行または別の文章が挟まる'
    a,b=speech_length(top),speech_length(bot)
    if a is None or b is None or (a>=low and b>=low):return False,'短文ではない'
    if a+b+len(gap)>high:return False,'結合後が長すぎる'
    return True,gap

def _safe_auto_split(s,low,high,used_ids):
    if s.get('segment_type','speech')!='speech' or speech_length(s) is None or speech_length(s)<=high:return [],''
    text=s.get('text')
    if not isinstance(text,str):return [],'原文なし'
    has_tts=isinstance(s.get('tts_text'),str) and bool(s['tts_text'].strip())
    if has_tts and s['tts_text']!=text:return [],'原文と異なる読み上げ文あり：手動で対応箇所を調整してください'
    fmt=s.get('text_format','none')
    whole,warnings=reading(text,fmt)
    if warnings:return [],'読み変換の確認が必要です'
    groups=[];rest=text;fallback=False
    # できるだけ均等な長さへ分け、極端に短い末尾を作らない。
    while len(reading(rest,fmt)[0])>high:
        count=max(2,math.ceil(len(reading(rest,fmt)[0])/high))
        k,weak=_split_choice(rest,len(rest)/count)
        if k is None:break
        a,b=rest[:k],rest[k:]
        ar,aw=reading(a,fmt);br,bw=reading(b,fmt);rr,rw=reading(rest,fmt)
        if aw or bw or rw or ar+br!=rr:break
        groups.append(a);rest=b;fallback=fallback or weak
    groups.append(rest)
    if len(groups)<2:return [],'ルビなどのまとまりを壊さずに分割できません'
    if ''.join(groups)!=text or ''.join(reading(x,fmt)[0] for x in groups)!=whole:
        return [],'原文または読みの連結が一致しません'
    result=[]
    for i,part in enumerate(groups):
        ns=copy.deepcopy(s);ns['text']=part
        if has_tts:ns['tts_text']=part
        if i:
            suffix=i+1;candidate=f"{s['id']}_part{suffix}"
            while candidate in used_ids:
                suffix+=1;candidate=f"{s['id']}_part{suffix}"
            ns['id']=candidate;used_ids.add(candidate)
        if i<len(groups)-1:
            ns.pop('pause_after_ms',None);ns.pop('blank_lines_after',None)
        mark_review(ns,'split',fallback)
        result.append(ns)
    return result,''


def _force_length_adjust(project,low,high):
    """利用者が保全検査の省略を選んだ場合。本文と読みの対応は推定する。"""
    candidate=copy.deepcopy(project);expanded=[];logs=[];skips=[]
    used={x['id'] for x in candidate['segments']}
    for original in candidate['segments']:
        text=original.get('text')
        if original.get('segment_type','speech')!='speech' or not isinstance(text,str) or not text:
            expanded.append(original);continue
        size=speech_length(original)
        if size is None or size<=high:
            expanded.append(original);continue
        # 通常のルビ範囲・読み一致チェックを通さず、原文の長さに割り振る。
        spoken=original.get('tts_text')
        if not isinstance(spoken,str) or not spoken.strip():
            try:spoken=reading(text,original.get('text_format','none'))[0]
            except Exception:spoken=text
        count=max(2,math.ceil(max(len(text),len(spoken))/high))
        import regex
        boundaries=[m.end() for m in regex.finditer(r'\X',text)]
        cuts=[0];weak_split=False
        for i in range(1,min(count,len(boundaries))):
            rest=text[cuts[-1]:]
            cut,weak=_split_choice(rest,len(rest)/(count-i+1),allow_unsafe=True)
            if cut is None:break
            cuts.append(cuts[-1]+cut);weak_split=weak_split or weak
        cuts.append(len(text))
        if len(cuts)<3:
            expanded.append(original);skips.append(f"{original['id']}: 分割可能な文字境界なし");continue
        reading_cuts=[round(len(spoken)*k/len(text)) for k in cuts]
        for i,(a,b) in enumerate(zip(cuts,cuts[1:])):
            part=copy.deepcopy(original);part['text']=text[a:b]
            if i:
                ident=f"{original['id']}_part{i+1}"
                while ident in used:ident+='_' 
                used.add(ident);part['id']=ident
            if original.get('tts_text') or original.get('text_format','none')!='none':
                part['tts_text']=spoken[reading_cuts[i]:reading_cuts[i+1]]
            if i<len(cuts)-2:
                part.pop('pause_after_ms',None);part.pop('blank_lines_after',None)
            mark_review(part,'split',weak_split)
            part['_extras']['koebuta_edit']['forced']=True
            expanded.append(part)
        logs.append(f"強制分割 {original['id']}: {len(cuts)-1}件（原文と読みの対応は推定）")
    candidate['segments']=expanded
    i=0
    while i<len(candidate['segments'])-1:
        a,b=candidate['segments'][i:i+2]
        if any(x.get('segment_type','speech')!='speech' or not isinstance(x.get('text'),str) or not x.get('text') for x in (a,b)):
            i+=1;continue
        la,lb=speech_length(a),speech_length(b)
        if la is None or lb is None or (la>=low and lb>=low) or la+lb>high:
            i+=1;continue
        gap=_source_gap_for_pair(candidate,i) if isinstance(candidate.get('source'),str) else ''
        gap=gap if isinstance(gap,str) else ''
        merged=copy.deepcopy(a);merged['text']=a['text']+gap+b['text']
        if a.get('tts_text') or b.get('tts_text') or a.get('text_format','none')!=b.get('text_format','none'):
            def spoken(x):
                if isinstance(x.get('tts_text'),str) and x['tts_text'].strip():return x['tts_text']
                try:return reading(x['text'],x.get('text_format','none'))[0]
                except Exception:return x['text']
            merged['tts_text']=spoken(a)+gap+spoken(b)
        for field in ('pause_after_ms','blank_lines_after'):
            if field in b:merged[field]=b[field]
            else:merged.pop(field,None)
        inherit_review(merged,b)
        mark_review(merged,'merge');merged['_extras']['koebuta_edit']['forced']=True
        candidate['segments'][i:i+2]=[merged]
        logs.append(f"強制結合 {a['id']} + {b['id']}（話者・感情・演技は前の行、間は後ろの行を採用）")
    return candidate,logs,skips

def _propose_length_adjust(project,low,high,force=False):
    if force:return _force_length_adjust(project,low,high)
    if not 1<=low<high:raise ValueError('文字数の範囲が不正です')
    candidate=copy.deepcopy(project)
    logs=[];skips=[];used={s['id'] for s in candidate['segments']}
    # 最初に長文を処理し、次に短文を統合。原稿と作品順序を保持する。
    expanded=[]
    for s in candidate['segments']:
        parts,reason=_safe_auto_split(s,low,high,used)
        if parts:
            expanded.extend(parts);logs.append(f"分割 {s['id']}: {len(parts)}件")
        else:
            expanded.append(s)
            if reason:skips.append(f"{s['id']}: {reason}")
    candidate['segments']=expanded
    i=0
    while i<len(candidate['segments'])-1:
        ok,gap=_safe_auto_merge(candidate,i,low,high)
        if not ok:
            i+=1;continue
        a,b=candidate['segments'][i:i+2]
        merged=copy.deepcopy(a);merged['text']=a['text']+gap+b['text']
        inherit_review(merged,b)
        mark_review(merged,'merge')
        for k in ('pause_after_ms','blank_lines_after'):
            if k in b:merged[k]=b[k]
            else:merged.pop(k,None)
        candidate['segments'][i:i+2]=[merged]
        logs.append(f"結合 {a['id']} + {b['id']}")
        # 同じ位置で次の短文とも結合できるか再確認
    return candidate,logs,skips

def set_length_limits(low,high,start,count,df=None):
    low,high=int(low),int(high)
    STATE.pop('length_approval',None)
    if not 1<=low<high:
        return (df if df is not None else render_table(start,count)[0]),'基準を確認してください','','⚠️ 短文基準は長文基準より小さくしてください'
    STATE['length_min'],STATE['length_max']=low,high
    if df is not None:
        try:draft_guard(lambda:None)(df)
        except gr.Error as error:
            # 検証失敗では入力を保持し、古い台本の候補を表示しない。
            return df,'入力内容を保持しています','',f'候補を更新できません。表の入力を修正してください：{error}'
    data,status,page=render_table(start,count)
    return data,status,page,preview_length_adjust()


def preview_length_adjust():
    project=STATE.get('project')
    if not project:return 'シナリオを読み込むと、発話の長さを確認できます。'
    low,high=STATE.get('length_min',12),STATE.get('length_max',140)
    short=sum(s.get('segment_type','speech')=='speech' and speech_length(s) is not None and speech_length(s)<low for s in project['segments'])
    long=sum(s.get('segment_type','speech')=='speech' and speech_length(s) is not None and speech_length(s)>high for s in project['segments'])
    STATE['length_approval']=(project_stamp(),low,high,bool(STATE.get('force_length',False)))
    _,changes,skips=_propose_length_adjust(project,low,high,bool(STATE.get('force_length',False)))
    lines=[f'**確認対象：短文 {short}件／長文 {long}件**（{low}文字未満・{high}文字超）',
           f'提案できる編集：{len(changes)}件。**自動適用はされません。**']
    if STATE.get('force_length',False):lines.append('⚠ 強制処理：原文照合・ルビ・読みの対応・話者や演技の違い・間の保全チェックを省略します。結果が壊れる場合があります。')
    if changes:lines+=['','\n'.join('- '+x for x in changes[:15])]
    for i,segment in enumerate(project['segments'] if not STATE.get('force_length',False) else []):
        size=speech_length(segment)
        if segment.get('segment_type','speech')=='speech' and size is not None and size<low:
            ok,reason=_safe_auto_merge(project,i,low,high)
            if not ok:skips.append(f"{segment['id']}: 次の行と結合しない理由：{reason}")
    if skips:lines+=['','変更を見送る条件（短文は次の行との判定）：','\n'.join('- '+x for x in skips[:10])]
    return '\n'.join(lines)

def apply_length_adjust(confirmed,start,count):
    if not confirmed:
        p=STATE.get('project')
        df,info,page=render_table(start,count)
        return '⚠️ 一括調整の注意事項を読み、確認チェックを入れてください。変更していません',df,info,page,issues_message(p),preview_length_adjust()
    p=STATE.get('project')
    if not p:
        df,info,page=render_table(start,count)
        return '台本を読み込んでください',df,info,page,issues_message(p),preview_length_adjust()
    try:
        signature=(project_stamp(),STATE.get('length_min',12),STATE.get('length_max',140),bool(STATE.get('force_length',False)))
        if STATE.get('length_approval')!=signature:
            raise ValueError('台本または条件が変わりました。候補を再確認してください')
        proposed,changes,skips=_propose_length_adjust(p,STATE.get('length_min',12),STATE.get('length_max',140),bool(STATE.get('force_length',False)))
        if not changes:
            msg='安全に自動調整できる発話はありません。長さの警告を見て手動で確認してください'
        else:
            # 元原稿の保存、発話ID重複、既存エラー数が増えないことを確認
            if proposed.get('source')!=p.get('source'):
                raise ValueError('原稿全文が変更されるため適用を中止しました')
            if len({s['id'] for s in proposed['segments']})!=len(proposed['segments']):
                raise ValueError('発話IDが重複したため適用を中止しました')
            if not STATE.get('force_length',False) and len(validate_project(proposed))>len(validate_project(p)):
                raise ValueError('確認事項が増えたため適用を中止しました')
            push_undo();STATE['project']=proposed;STATE['last_files']=[]
            msg=f'{len(changes)}件の分割・結合を適用しました。音声生成前に必ず台本を確認してください'
        df,info,page=render_table(start,count)
        return msg,df,info,page,issues_message(STATE.get('project')),preview_length_adjust()
    except Exception as e:
        df,info,page=render_table(start,count)
        return f'自動調整を適用できません: {e}',df,info,page,issues_message(p),preview_length_adjust()

# --- 行の削除 ---
def delete_single_row(target_idx, start, count):
    p = STATE.get('project')
    if not p or not p.get('segments'):
        return 'シナリオがありません', *render_table(start, count)
    try:
        idx = int(target_idx)
        if not (0 <= idx < len(p['segments'])):
            return f'行番号 {idx} が範囲外です', *render_table(start, count)
        push_undo()
        del_item = p['segments'].pop(idx)
        STATE['last_files'] = []
        msg = f'行 #{idx}（id: {del_item.get("id")}）を削除しました'
        df_out, _, page_msg = render_table(start, count)
        return msg, df_out, msg, page_msg
    except Exception as e:
        df_out, _, page_msg = render_table(start, count)
        return f'削除失敗: {e}', df_out, f'エラー: {e}', page_msg

# --- 単一行の即時試聴 / 生成 ---
def preview_single_row(target_idx, seed, steps, use_acting, use_emotion, narrator_emo):
    p = STATE.get('project')
    if not p or not p.get('segments'):
        return 'シナリオが未読込です', None
    try:
        idx = int(target_idx)
        if not (0 <= idx < len(p['segments'])):
            return f'行番号 {idx} が不正です', None
        seg = p['segments'][idx]
        
        # 計画を1発話のみ作成（長文は split_text で複数ユニットに分割されるため全部合成）
        sub_project = copy.deepcopy(p)
        sub_project['segments'] = [seg]
        sub_project['roles'] = [r for r in p['roles'] if r['id'] == seg['role']] or sub_project['roles']
        plan = build_plan(sub_project, STATE.get('voices', {}),
                          use_acting=bool(use_acting), use_emotion=bool(use_emotion),
                          narrator_emotion=bool(narrator_emo), seed=int(seed), steps=int(steps), global_pause_ms=280)
        
        runtime = load_runtime()
        chunks, sr_out = [], None
        for unit in plan:
            audio, sr = synthesize(unit, runtime)
            if sr_out is None: sr_out = sr
            chunks.append(audio)
            # キャッシュにも登録
            key = request_key(unit, MODEL_ID)
            cache_wav = cache_path(unit['unit_id'], key)
            sf.write(str(cache_wav), audio, sr, subtype='PCM_16')
            cache_wav.with_suffix('.json').write_text(json.dumps({'key':key, 'unit_id':unit['unit_id'], 'segment_id':unit['segment_id']},ensure_ascii=False), encoding='utf-8')
        
        import numpy as np
        combined = np.concatenate(chunks) if len(chunks) > 1 else chunks[0]
        temp_wav = session_out_path(f'single_preview_{hashlib.sha256(seg["id"].encode("utf-8")).hexdigest()[:24]}.wav')
        sf.write(str(temp_wav), combined, sr_out, subtype='PCM_16')
        
        unit_count = len(plan)
        return f'行 #{idx}（{unit_count}ユニット）の音声を生成・キャッシュしました', str(temp_wav)
    except Exception as e:
        return f'行 #{target_idx} 生成エラー: {e}', None

# --- 全体差分生成 ---
def run_master_generation(use_acting, use_emotion, narrator_emotion, seed, steps, pause_sec, progress=gr.Progress()):
    p = STATE.get('project')
    if not p:
        yield 'シナリオが未読込です', None, None
        return
    stamp=project_stamp()
    p=copy.deepcopy(p)
    voices=copy.deepcopy(STATE.get('voices',{}))
    logs = []
    STATE['last_files'] = []
    try:
        plan = build_plan(p, voices, use_acting=bool(use_acting), use_emotion=bool(use_emotion),
                          narrator_emotion=bool(narrator_emotion), seed=int(seed), steps=int(steps), global_pause_ms=int(float(pause_sec) * 1000))
        logs.append(f'【スタジオ】全 {len(plan)} ユニットの合成計画を策定しました')
        for w in STATE.get('plan_warnings', []):
            logs.append('⚠️ 短い発話: ' + w)
        yield '\n'.join(logs), None, None
        
        if project_stamp()!=stamp:raise ValueError('台本が変わったため旧作品の生成を中止しました')
        runtime = load_runtime()
        files = []
        cached_count, gen_count = 0, 0
        
        for n, unit in enumerate(plan, 1):
            if project_stamp()!=stamp:raise ValueError('台本が変わったため生成を中止しました')
            progress(n / len(plan), desc=f'[{n}/{len(plan)}] {unit["unit_id"]}')
            key = request_key(unit, MODEL_ID)
            wav = cache_path(unit['unit_id'], key)
            meta = wav.with_suffix('.json')
            
            if wav.exists() and meta.exists() and json.loads(meta.read_text(encoding='utf-8')).get('key') == key:
                cached_count += 1
                logs.append(f'[{n}/{len(plan)}] {unit["unit_id"]}: キャッシュを再利用')
            else:
                gen_count += 1
                logs.append(f'[{n}/{len(plan)}] {unit["unit_id"]}: 新規音声を生成中...')
                yield '\n'.join(logs[-12:]), None, None
                if project_stamp()!=stamp:raise ValueError('台本が変わったため生成を中止しました')
                audio, sr = synthesize(unit, runtime)
                sf.write(str(wav), audio, sr, subtype='PCM_16')
                meta.write_text(json.dumps({'key':key, 'unit_id':unit['unit_id'], 'segment_id':unit['segment_id']},ensure_ascii=False), encoding='utf-8')
            files.append((unit, wav))
            
        if project_stamp()!=stamp:raise ValueError('台本が変わったため生成結果を破棄しました')
        out_wav = concat(files)
        if project_stamp()!=stamp:raise ValueError('台本が変わったため生成結果を破棄しました')
        STATE['last_files'] = files
        logs.append(f'\n🎉 全編の結合が完了しました！（新規生成: {gen_count}件 / キャッシュ利用: {cached_count}件）')
        yield '\n'.join(logs[-12:]), str(out_wav), str(out_wav)
    except Exception as e:
        logs.append(f'\n❌ 生成中断: {e}')
        yield '\n'.join(logs), None, None

def session_out_path(filename):
    # セッション固有の成果物名にする（他ユーザーとファイル名が衝突しない）。
    import uuid
    session = STATE._data()
    token = session.setdefault('_file_token', uuid.uuid4().hex)
    directory = OUT_DIR / token
    directory.mkdir(parents=True,exist_ok=True)
    return directory / filename

def save_scenario_json():
    p = STATE.get('project')
    if not p: return None
    try:
        out = export_scenario(p)
        path = session_out_path('scenario_edited.json')
        path.write_text(json.dumps(out,ensure_ascii=False,indent=2),encoding='utf-8')
        return str(path)
    except Exception as e:
        raise gr.Error(f'JSONの保存に失敗しました: {e}')



# -------- Gradioの状態はブラウザーセッション単位で分離する --------
import inspect

def session_event(fn):
    """状態入力を受け取り、状態を出力し直す（Gradioのセッション保管）。"""
    def called(session, *args):
        token = _CTX.set(session)
        try:
            result = fn(*args)
            if isinstance(result,tuple): return (*result, session)
            return result,session
        finally:
            _CTX.reset(token)
    sig = inspect.signature(fn)
    called.__signature__ = sig.replace(parameters=[inspect.Parameter('session',inspect.Parameter.POSITIONAL_OR_KEYWORD),*sig.parameters.values()])
    called.__name__ = 'session_' + fn.__name__
    return called

def session_generate(session, use_acting, use_emotion, narrator_emotion, seed, steps, pause_sec, progress=gr.Progress()):
    """Gradioの進捗イベントを跨いでContextVarのトークンを保持しない。

    各next()呼出しの間だけセッションを有効にする。Gradioがジェネレータを
    異なる実行コンテキストで再開しても、Tokenの解除で失敗しない。
    """
    generator = run_master_generation(use_acting, use_emotion, narrator_emotion,
                                      seed, steps, pause_sec, progress=progress)
    while True:
        token = _CTX.set(session)
        try:
            try:
                result = next(generator)
            except StopIteration:
                return
        finally:
            _CTX.reset(token)
        yield (*result, session)

def current_issues():
    return issues_message(STATE.get('project'))

def confirm_review(target_idx,start,count):
    p=STATE.get('project')
    idx=int(target_idx)
    if not p or not 0<=idx<len(p['segments']):return '行番号を確認してください',*render_table(start,count)
    meta=p['segments'][idx].get('_extras',{}).get('koebuta_edit')
    if isinstance(meta,dict):
        push_undo();meta['needs_review']=False
    return 'この行を確認済みにしました',*render_table(start,count)

def sync_role_ui(role_id=None):
    choices=get_role_choices()
    selected=role_id if role_id in [x[1] for x in choices] else (choices[0][1] if choices else None)
    return gr.update(choices=choices,value=selected),*on_role_change(selected)

def draft_guard(fn):
    # 表を再描画する操作の前に入力を保存。エラーなら画面を変更せず中止。
    def guarded(df,*args):
        view=STATE.get('table_view')
        if view and df is not None:
            columns=[c for c in EDIT_COLUMNS if c not in ('#','長さの確認')]
            def canonical(frame):
                frame=frame[columns].copy().fillna('')
                try:
                    frame['感情（絵文字・名前）']=frame['感情（絵文字・名前）'].map(parse_emotion)
                    frame['発話後の間(ms)']=frame['発話後の間(ms)'].map(lambda x: '' if _pause_value(x) is None else str(_pause_value(x)))
                except ValueError: pass
                return frame.astype(str).reset_index(drop=True)
            current=canonical(df)
            saved=canonical(view['data'])
            if not current.equals(saved):
                result=save_table_edits(df,view['start'],view['count'])
                if result[0].startswith('保存できません'):raise gr.Error(result[0])
        return fn(*args)
    sig=inspect.signature(fn)
    guarded.__signature__=sig.replace(parameters=[inspect.Parameter('_draft_table',inspect.Parameter.POSITIONAL_OR_KEYWORD),*sig.parameters.values()])
    guarded.__name__='draft_'+fn.__name__
    return guarded

def editor_ui(fn):
    # 編集と関連する画面を一つのイベント結果として同期する。
    def called(df,*args):
        result=fn(*args) if fn in (save_table_edits,undo_action) else draft_guard(fn)(df,*args)
        if isinstance(result,tuple) and str(result[0]).startswith(('保存できません','読み込み失敗')):
            raise gr.Error(result[0])
        view=STATE.get('table_view') or {}
        start,count=view.get('start',0),view.get('count',50)
        data,_,page=render_table(start,count)
        message=('表を更新しました' if fn is render_table else (result[0] if isinstance(result,tuple) else result))
        role_id=None
        if fn is save_role_action:role_id=args[1]
        return (message,project_summary(),data,message,page,issues_message(STATE.get('project')),
                *sync_role_ui(role_id),None,None,None,None,None,None,None,False,start,count)
    sig=inspect.signature(fn)
    called.__signature__=sig.replace(parameters=[inspect.Parameter('_draft_table',inspect.Parameter.POSITIONAL_OR_KEYWORD),*sig.parameters.values()])
    called.__name__='editor_'+fn.__name__
    return called

def session_generate_ui(session,df,use_acting,use_emotion,narrator_emotion,seed,steps,pause_sec,progress=gr.Progress()):
    token=_CTX.set(session)
    try: draft_guard(lambda:None)(df)
    finally: _CTX.reset(token)
    yield from session_generate(session,use_acting,use_emotion,narrator_emotion,seed,steps,pause_sec,progress)

def preview_length_ui(df):
    message=draft_guard(preview_length_adjust)(df)
    view=STATE.get('table_view') or {}
    data,status,page=render_table(view.get('start',0),view.get('count',50))
    return message,data,status,page,False

def set_force_length(value,df):
    STATE['force_length']=bool(value)
    STATE.pop('length_approval',None)
    return preview_length_ui(df)

def set_length_ui(low,high,start,count,df):
    return (*set_length_limits(low,high,start,count,df),False)

# ==================== Gradio UI 構築 ====================
custom_css = """
.studio-box { border-radius: 8px; padding: 12px; margin-bottom: 12px; border: 1px solid #e0e0e0; }
.action-bar { background: #f8fafc; border-radius: 6px; padding: 8px; margin-bottom: 8px; }
"""

with gr.Blocks(title='Koebuta Studio v1.00') as demo:
    session_store = gr.State(new_session())
    gr.Markdown('## 🎙️ Koebuta 朗読スタジオ — v1.00\nシナリオの確認・話者ごとの声の設定・朗読音声の生成')
    gr.Markdown('**⚠️ Colab無料枠での外部Web UI利用には制限があります。** ノートブックUIを迂回して主にGradioで操作する用途は、ランタイムが予告なく終了する場合があります。Colabでは有料プランかつコンピューティング単位が残っている状態での利用を推奨します。詳細はノートブック冒頭の規約・権利・透かしに関する注意をご確認ください。')
    
    with gr.Row():
        # ========== 左カラム：プロジェクト & キャスト管理 (28%) ==========
        with gr.Column(scale=3, min_width=320):
            with gr.Accordion('📁 1. 台本の読み込み・保存', open=True):
                load_file = gr.File(label='シナリオJSONファイル', file_types=['.json'])
                load_paste = gr.Textbox(label='またはJSONを貼り付け', lines=2, placeholder='{"schema_version": 1, "speakers": {...}, "segments": [...]}')
                reading_mode = gr.Radio(choices=[('全員が同じ声','shared'),('話者ごとに別の声','cast')], label='朗読方式', value='cast',
                                        info='１人の朗読者で読む場合は「全員が同じ声」を選びます')
                load_btn = gr.Button('📥 シナリオを読み込む', variant='primary')
                load_status_txt = gr.Textbox(label='読込結果', interactive=False, max_lines=2)
                save_json_btn = gr.Button('💾 編集後シナリオJSONを出力')
                save_json_file = gr.File(label='シナリオJSONの保存')
                issue_box = gr.Textbox(label='読み込み時の注意・生成前の確認事項',interactive=False,lines=5,max_lines=12)
                
            with gr.Accordion('🎭 2. 話者と声の設定', open=True):
                char_select = gr.Dropdown([], label='設定する話者（同名でもIDで区別）')
                with gr.Accordion('話者の情報を確認・修正',open=False):
                    role_id_input = gr.Textbox(label='話者ID（他の話者と重複しない）')
                    role_name_input = gr.Textbox(label='話者名（画面に表示する名前）')
                    role_kind_input = gr.Dropdown(choices=[('地の文','narrator'),('登場人物','character')],label='話者の種類')
                    role_bio_input = gr.Textbox(label='人物像',lines=3)
                    role_acting_input = gr.Textbox(label='普段の話し方',lines=3)
                    gr.Markdown('「声の特徴」は下の「声を試作」タブにあります。話者を修正したら、必ず保存してください。')
                    save_role_btn = gr.Button('話者の情報を保存')
                active_wav_preview = gr.Audio(label='現在割り当てられている音声', type='filepath', interactive=False)
                
                with gr.Tab('✨ 声を試作'):
                    cap_input = gr.Textbox(label='声の特徴（試作音声に反映）', lines=2)
                    sample_sample_txt = gr.Textbox(label='試し読みの文章（短すぎない文章を推奨）', value=SAMPLE_TEXT_DEFAULT, lines=2)
                    with gr.Row():
                        s_seed = gr.Number(label='生成シード', value=314, precision=0, scale=2)
                        s_rand = gr.Button('🎲', scale=1)
                        s_steps = gr.Dropdown([20, 40, 60], label='生成ステップ数', value=40, scale=2)
                    make_sample_btn = gr.Button('試作音声を生成', variant='secondary')
                    sample_audio_out = gr.Audio(label='試作音声の試聴', type='filepath')
                    adopt_sample_btn = gr.Button('✅ この試作音声を話者に割り当てる', variant='primary')
                    
                with gr.Tab('📂 音声ファイルを使用'):
                    upload_wav = gr.Audio(label='参照音声をアップロード（WAV）', type='filepath', sources=['upload'])
                    adopt_upload_btn = gr.Button('✅ この参照音声を話者に割り当てる')
                    
                with gr.Tab('🎤 声を録音'):
                    mic_wav = gr.Audio(label='参照音声を録音', type='filepath', sources=['microphone'])
                    adopt_mic_btn = gr.Button('✅ 録音した音声を話者に割り当てる')
                    
                download_refs_btn = gr.Button('📦 設定した参照音声をZIP保存')
                refs_zip_file = gr.File(label='参照音声のZIP')
                
                    
        # ========== 中央カラム：台本・タイムラインエディタ (48%) ==========
        with gr.Column(scale=5, min_width=500):
            summary_banner = gr.Textbox(label='📊 シナリオの状態', value='シナリオ未読込', interactive=False, max_lines=4)
            
            with gr.Row():
                page_start = gr.Number(label='表示開始行', value=0, precision=0, scale=2)
                page_count = gr.Dropdown([20, 50, 100], label='表示件数', value=50, scale=2)
                page_refresh_btn = gr.Button('🔄 更新', scale=1)
                page_info_txt = gr.Markdown('全 0 件')
                
            script_table = gr.Dataframe(
                headers=EDIT_COLUMNS,
                interactive=True,
                type='pandas',
                wrap=True,
                column_widths=TABLE_WIDTHS,
                max_height=520
            )
            
            with gr.Row():
                table_save_btn = gr.Button('💾 表の編集を反映', variant='primary', scale=3)
                undo_btn = gr.Button('↩️ 1つ前に戻す', scale=1)
                
            with gr.Row():
                table_status = gr.Textbox(label='編集結果', interactive=False, scale=4)
                
            with gr.Accordion('📏 発話の長さを確認・調整', open=True):
                gr.Markdown('**赤い警告は目安です。** 声・話速・演技で適切な長さは変わります。初期状態では原稿も発話も変更しません。分割は文末・改行・読点などを優先し、候補がなければ文字数付近で分けます。自動結合は同じ話者・感情などの条件を満たす短文が対象です。')
                with gr.Row():
                    min_length_slider = gr.Slider(5,40,value=12,step=1,label='短文の警告（この文字数未満）')
                    max_length_slider = gr.Slider(80,240,value=140,step=10,label='長文の警告（この文字数超）')
                length_preview_btn = gr.Button('🔎 分割・結合の候補を確認')
                length_preview_box = gr.Markdown('シナリオ読み込み後、候補を確認できます。')
                gr.Markdown('⚠️ **一括調整すると、警告対象のうち安全に処理できる発話の区切りとIDが変わります。** 短文は条件を満たす隣と結合、長文はなるべく自然な位置で分割します。ルビや絵文字は途中で切りません。原文と異なる読み上げ文は手動で調整してください。変更行には要確認の印が残ります。原文の文字は消しませんが、音声・演技の区切りが変わるため、適用後は試聴してください。**変更はUndoで最大10操作戻せます。**')
                force_length_check = gr.Checkbox(label='⚠ 警告を無視して強制分割・結合する（原文・読みが壊れる可能性を許容）',value=False)
                gr.Markdown('有効時は原文照合・ルビ・読みの対応・話者・感情・演技・間の保全条件も省略します。結合は前の行の話者・感情・演技を採用し、読みの分割位置は推定します。適用後は必ず確認し、必要ならUndoしてください。区切り行は維持し、文字数条件は引き続き使用します。')
                length_confirm = gr.Checkbox(label='候補と注意事項を確認しました（一括調整を許可）',value=False)
                length_apply_btn = gr.Button('⚠️ 警告対象を一括で分割・結合（変更可能な箇所のみ）',variant='secondary')
                gr.Markdown('※入力中の表は操作前に保存します。入力エラーがあれば保持して中止します。原文と異なる読み上げ文は対応箇所を特定できないため手動調整が必要です。変更後は要確認の行を確認してください。')

            with gr.Accordion('🛠️ 発話の操作（結合・削除・試聴）', open=True):
                gr.Markdown('※感情は絵文字・日本語で表示しますがJSONにはIDを保存します。**個別結合は、異なる感情・演技指定がある場合、先頭の発話を優先**します（結合結果に警告します）。個別分割は感情・演技指示を両方へ引き継ぎます。\n※**原文と異なる読み上げ文がある発話は、対応箇所を安全に判断できないため個別分割できません。** 発話の長さ警告は目安です。\n※操作対象の番号は表の「#」列と同じ**0始まり**です（45件目は44）。未保存の表編集は先に「表の編集を反映」を押してください。')
                with gr.Row():
                    target_row_num = gr.Number(label='操作対象の行番号（0から）', value=0, precision=0, scale=2)
                    merge_btn = gr.Button('🔗 次の発話と結合', variant='secondary', scale=3)
                    delete_btn = gr.Button('🗑️ この発話を削除', scale=2)
                    preview_row_btn = gr.Button('▶️ この発話を生成して試聴', variant='primary', scale=3)
                with gr.Row():
                    split_at = gr.Number(label='分割位置（省略時は自然な位置を優先・先頭からの文字数）',value=None,precision=0,scale=2)
                    split_btn = gr.Button('✂️ この発話を分割',variant='secondary',scale=3)
                review_done_btn = gr.Button('✅ この行を確認済みにする')
                gr.Markdown('長さの確認列に分割・結合の履歴と要確認を表示します。「自然な区切りなし」の行は特に読みのつながりを確認してください。確認済みの操作もUndoできます。')
                row_audio_preview = gr.Audio(label='発話の試聴', type='filepath')
                
        # ========== 右カラム：マスター出力 & レンダリング (24%) ==========
        with gr.Column(scale=3, min_width=280):
            # 毎回確認する設定は右カラム最上部に常時表示。折りたたみには入れない。
            gr.Markdown('### ⚙️ 3. 音声生成の設定')
            gr.Markdown('感情・演技の反映は音質に影響します。**地の文の感情は初期状態で無効**です。')
            opt_acting = gr.Checkbox(label='演技指示を音声に反映', value=False)
            opt_emotion = gr.Checkbox(label='感情の絵文字を音声に反映', value=True)
            opt_narrator_emo = gr.Checkbox(label='地の文にも感情の絵文字を反映', value=False)
            g_seed = gr.Number(label='共通の生成シード', value=314, precision=0)
            g_steps = gr.Dropdown([20, 40, 60], label='生成ステップ数', value=40)
            g_pause = gr.Number(label='標準の発話後の間（秒）', value=0.28)
            gr.Markdown('---')
            gr.Markdown('### 🎬 4. 朗読音声の生成')
            render_master_btn = gr.Button('⚡ 変更箇所だけ再生成して全編を結合', variant='primary', size='lg')
            master_log_box = gr.Textbox(label='音声生成の進捗', lines=8, interactive=False)
            master_player = gr.Audio(label='完成した朗読音声（WAV）', type='filepath')
            master_download = gr.File(label='完成音声の保存')
            
            gr.Markdown('---')
            gr.Markdown('### 📦 個別音声の保存')
            gr.Markdown('※直近の全編生成に使用した個別音声をZIPで保存します。結合音声とは別に保管できます。')
            download_segs_btn = gr.Button('📦 直近の個別音声をZIPで保存')
            segs_zip_file = gr.File(label='個別音声のZIP')

    # 一つの編集イベントで表・話者フォーム・音声出力を同期する。
    editor_outputs=[load_status_txt,summary_banner,script_table,table_status,page_info_txt,issue_box,
                    char_select,active_wav_preview,cap_input,role_id_input,role_name_input,role_kind_input,role_bio_input,role_acting_input,
                    master_player,master_download,segs_zip_file,row_audio_preview,save_json_file,refs_zip_file,sample_audio_out,
                    length_confirm,page_start,page_count,session_store]
    # セッション操作と生成を同一キューへ直列化。複数セッション間も共有GPUを保護する。
    event_queue={'concurrency_id':'koebuta_studio','concurrency_limit':1}
    load_btn.click(session_event(editor_ui(load_scenario_action)),
        inputs=[session_store,script_table,load_file,load_paste,reading_mode],outputs=editor_outputs,**event_queue)
    save_json_btn.click(session_event(draft_guard(save_scenario_json)),inputs=[session_store,script_table],outputs=[save_json_file,session_store],**event_queue)
    char_select.change(session_event(on_role_change),inputs=[session_store,char_select],
        outputs=[active_wav_preview,cap_input,role_id_input,role_name_input,role_kind_input,role_bio_input,role_acting_input,session_store],**event_queue)
    save_role_btn.click(session_event(editor_ui(save_role_action)),
        inputs=[session_store,script_table,char_select,role_id_input,role_name_input,role_kind_input,role_bio_input,cap_input,role_acting_input],outputs=editor_outputs,**event_queue)
    s_rand.click(lambda: random.randint(0,2147483647),outputs=s_seed,**event_queue)
    make_sample_btn.click(session_event(draft_guard(gen_sample_action)),inputs=[session_store,script_table,char_select,cap_input,sample_sample_txt,s_seed,s_steps],outputs=[table_status,sample_audio_out,session_store],**event_queue)
    # 参照WAV採用もUndo履歴へ保存。フォームと完成音声も更新する。
    for button,fn,args in [(adopt_sample_btn,adopt_sample_action,[char_select]),
                           (adopt_upload_btn,adopt_upload_action,[char_select,upload_wav]),
                           (adopt_mic_btn,adopt_upload_action,[char_select,mic_wav])]:
        button.click(session_event(editor_ui(fn)),inputs=[session_store,script_table,*args],outputs=editor_outputs,**event_queue)
    download_refs_btn.click(session_event(download_refs_zip),inputs=[session_store],outputs=[refs_zip_file,session_store],**event_queue)
    page_refresh_btn.click(session_event(editor_ui(render_table)),inputs=[session_store,script_table,page_start,page_count],outputs=editor_outputs,**event_queue)
    table_save_btn.click(session_event(editor_ui(save_table_edits)),inputs=[session_store,script_table,script_table,page_start,page_count],outputs=editor_outputs,**event_queue)
    undo_btn.click(session_event(editor_ui(undo_action)),inputs=[session_store,script_table,page_start,page_count],outputs=editor_outputs,**event_queue)
    for ctrl in (min_length_slider,max_length_slider):
        ctrl.change(session_event(set_length_ui),inputs=[session_store,min_length_slider,max_length_slider,page_start,page_count,script_table],
            outputs=[script_table,table_status,page_info_txt,length_preview_box,length_confirm,session_store],**event_queue)
    force_length_check.change(session_event(set_force_length),inputs=[session_store,force_length_check,script_table],outputs=[length_preview_box,script_table,table_status,page_info_txt,length_confirm,session_store],**event_queue)
    length_preview_btn.click(session_event(preview_length_ui),inputs=[session_store,script_table],outputs=[length_preview_box,script_table,table_status,page_info_txt,length_confirm,session_store],**event_queue)
    length_apply_btn.click(session_event(editor_ui(apply_length_adjust)),inputs=[session_store,script_table,length_confirm,page_start,page_count],outputs=editor_outputs,**event_queue)
    merge_btn.click(session_event(editor_ui(merge_row_with_next)),inputs=[session_store,script_table,target_row_num,page_start,page_count],outputs=editor_outputs,**event_queue)
    split_btn.click(session_event(editor_ui(split_single_row)),inputs=[session_store,script_table,target_row_num,split_at,page_start,page_count],outputs=editor_outputs,**event_queue)
    delete_btn.click(session_event(editor_ui(delete_single_row)),inputs=[session_store,script_table,target_row_num,page_start,page_count],outputs=editor_outputs,**event_queue)
    review_done_btn.click(session_event(editor_ui(confirm_review)),inputs=[session_store,script_table,target_row_num,page_start,page_count],outputs=editor_outputs,**event_queue)
    preview_row_btn.click(session_event(draft_guard(preview_single_row)),inputs=[session_store,script_table,target_row_num,g_seed,g_steps,opt_acting,opt_emotion,opt_narrator_emo],outputs=[table_status,row_audio_preview,session_store],**event_queue)
    render_master_btn.click(session_generate_ui,inputs=[session_store,script_table,opt_acting,opt_emotion,opt_narrator_emo,g_seed,g_steps,g_pause],outputs=[master_log_box,master_player,master_download,session_store],**event_queue)
    download_segs_btn.click(session_event(download_segments_zip),inputs=[session_store],outputs=[segs_zip_file,session_store],**event_queue)

# 公開リンクが知られてもシナリオ・声素材を閲覧されないよう、ランダムなログイン情報を発行する。
# COLABセルの出力にパスワードが表示されます。第三者に共有しないでください。
import secrets, os
ui_user = os.environ.get('KOEBUTA_UI_USER','koebuta')
ui_password = os.environ.get('KOEBUTA_UI_PASSWORD') or secrets.token_urlsafe(16)
print('Koebutaのログインユーザー:',ui_user)
print('Koebutaのログインパスワード:',ui_password)
demo.queue(default_concurrency_limit=1)
demo.launch(share=True, auth=(ui_user,ui_password), theme=gr.themes.Soft(), css=custom_css, show_error=True)


## シナリオ作成と読みの調整

1. `02_CHAT_PROMPT.md` の依頼文で、LLMと話者・台本構成について対話し、シナリオJSONを作成します。LLMにIrodori-TTS固有の内部事情を伝える必要はありません。
2. 古典作品や特殊なルビの多い作品だけ、必要なら **`03C_READING_ADDON.md`** を追加して読み上げ文（`tts_text`）の調整方針を相談します。通常の作品には不要です。
3. JSONを読み込み、警告があれば必要な発話を修正します。表を直接編集した場合は「表の編集を反映」を押してください。
4. 声の設定をして試聴し、全編を生成します。極端に短い発話や特殊な漢字の読みはとくに確認してください。
5. ランタイムが終了する前に、編集済みJSON、完成WAV、必要な個別音声ZIP・参照音声ZIPを保存してください。

**利用時の注意：** Colab無料枠のWeb UI利用制限、声の模倣と著作権、電子透かしについてはノートブック冒頭の重要事項を参照してください。

詳しい操作・用語・FAQは `USER_GUIDE_FAQ.md` を、ローカル移植やAI開発エージェントへの依頼方法は `LOCAL_DEVELOPMENT_HANDOFF.md` をご覧ください。
